In [ ]:
from __future__ import annotations

import re
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import csv
from jmetal.core.solution import FloatSolution

try:
    from jmetal.core.quality_indicator import HyperVolume, InvertedGenerationalDistance
except ImportError:
    from jmetal.component.quality_indicator import HyperVolume, InvertedGenerationalDistance

try:
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False


NSGA_LABEL = "NSGA-II"
RANDOM_LABEL = "Random"

ATTACK_COL = "Obj1_Attack_Score"
DIST_COL = "Obj2_Distortion_Score"
IMAGE_NAME_COL = "Image_Name"
IMAGE_INDEX_COL = "Image_Index"
GEN_COL = "Generation"

PLOT_DPI = 300


def drop_duplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.loc[:, ~df.columns.duplicated()].copy()

@dataclass
class FrontRecord:
    vlm: str
    run_id: str
    approach: str
    benchmark: str
    csv_paths: List[str]
    front_min: np.ndarray  # minimization space: [-attack, distortion]


def natural_key(text: str):
    return [int(tok) if tok.isdigit() else tok.lower() for tok in re.split(r"(\d+)", str(text))]


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def make_solution(objectives: Iterable[float]) -> FloatSolution:
    objectives = list(map(float, objectives))
    solution = FloatSolution(
        lower_bound=[-1e9, -1e9],
        upper_bound=[1e9, 1e9],
        number_of_objectives=2,
    )
    solution.objectives = objectives
    return solution


def points_to_solutions(points: np.ndarray) -> List[FloatSolution]:
    return [make_solution(row) for row in np.asarray(points, dtype=float)]


def nondominated_min(points: np.ndarray) -> np.ndarray:
    points = np.asarray(points, dtype=float)

    if points.size == 0:
        return np.empty((0, 2), dtype=float)

    keep = np.ones(len(points), dtype=bool)

    for i in range(len(points)):
        if not keep[i]:
            continue
        for j in range(len(points)):
            if i == j or not keep[j]:
                continue
            dominates = (
                points[j, 0] <= points[i, 0]
                and points[j, 1] <= points[i, 1]
                and (points[j, 0] < points[i, 0] or points[j, 1] < points[i, 1])
            )
            if dominates:
                keep[i] = False
                break

    nd = points[keep]
    nd = np.unique(np.round(nd, decimals=12), axis=0)
    return nd


def load_mapping(mapping_csv: Optional[Path]) -> Dict[str, str]:
    if mapping_csv is None:
        return {}

    df = pd.read_csv(mapping_csv,  engine="python", quoting=csv.QUOTE_MINIMAL, on_bad_lines="skip")
    required = {"Image_Index", "Image_Name"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Mapping CSV is missing columns: {sorted(missing)}")

    mapping = {}
    for _, row in df.iterrows():
        try:
            idx_key = str(int(float(row["Image_Index"])))
        except (ValueError, TypeError):
            idx_key = str(row["Image_Index"]).strip()
        mapping[idx_key] = str(row["Image_Name"]).strip()

    return mapping

def benchmark_key_from_row(row: pd.Series, mapping: Dict[str, str]) -> str:
    if IMAGE_NAME_COL in row.index:
        value = row[IMAGE_NAME_COL]
        if isinstance(value, pd.Series):
            value = value.iloc[0]
        if pd.notna(value):
            image_name = str(value).strip()
            if image_name != "":
                return image_name

    if IMAGE_INDEX_COL in row.index:
        value = row[IMAGE_INDEX_COL]
        if isinstance(value, pd.Series):
            value = value.iloc[0]
        if pd.notna(value):
            try:
                idx_key = str(int(float(value)))
            except (ValueError, TypeError):
                idx_key = str(value).strip()
            return mapping.get(idx_key, idx_key)

    raise ValueError("Neither Image_Name nor Image_Index found in CSV row.")

def detect_approach(csv_path: Path) -> Optional[str]:
    tokens = [csv_path.name.lower()] + [p.name.lower() for p in csv_path.parents]

    for token in tokens:
        if "random" in token:
            return RANDOM_LABEL
        if "nsgaii" in token or re.search(r"\bnsga\b", token):
            return NSGA_LABEL

    return None


def detect_fragment_kind(csv_path: Path, approach: str) -> str:
    name = csv_path.name.lower()

    if approach == RANDOM_LABEL:
        return "random"

    if "pareto" in name:
        return "pareto"
    if "population" in name:
        return "population"

    return "nsga_other"


def extract_run_id(csv_path: Path) -> str:
    for p in [csv_path.parent] + list(csv_path.parents):
        m = re.fullmatch(r"run_(\d+)", p.name.lower())
        if m:
            return f"run{int(m.group(1))}"
    return "run1"


def load_csv_fragment_fronts(
    csv_path: Path,
    mapping: Dict[str, str],
    approach: str,
) -> Dict[str, np.ndarray]:
    """
    Return benchmark -> front fragment in minimization space.

    Objective space used:
      f1 = -Attack_Score
      f2 = Distortion_Score
    """
    df = pd.read_csv(csv_path,  engine="python", quoting=csv.QUOTE_MINIMAL, on_bad_lines="skip")
    df = df.loc[:, ~df.columns.duplicated()].copy()

    required = {ATTACK_COL, DIST_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{csv_path} is missing required columns: {sorted(missing)}")

    fragment_kind = detect_fragment_kind(csv_path, approach)
    grouped_rows: Dict[str, List[Tuple[float, float]]] = defaultdict(list)

    # ---------- build benchmark column WITHOUT apply ----------
    if IMAGE_NAME_COL in df.columns:
        benchmark_col = df[IMAGE_NAME_COL]
        if isinstance(benchmark_col, pd.DataFrame):
            benchmark_col = benchmark_col.iloc[:, 0]
        benchmark_col = benchmark_col.astype(str).str.strip()

    elif IMAGE_INDEX_COL in df.columns:
        benchmark_col = df[IMAGE_INDEX_COL]
        if isinstance(benchmark_col, pd.DataFrame):
            benchmark_col = benchmark_col.iloc[:, 0]

        def map_idx(x):
            if pd.isna(x):
                return ""
            try:
                idx_key = str(int(float(x)))
            except (ValueError, TypeError):
                idx_key = str(x).strip()
            return mapping.get(idx_key, idx_key)

        benchmark_col = benchmark_col.map(map_idx).astype(str).str.strip()
    else:
        raise ValueError(f"{csv_path} must contain either '{IMAGE_NAME_COL}' or '{IMAGE_INDEX_COL}'.")

    df["_benchmark"] = benchmark_col

    # drop empty benchmark rows if any
    df = df[df["_benchmark"] != ""].copy()

    # ---------- NSGA-II population: keep only last generation ----------
    if approach == NSGA_LABEL and fragment_kind == "population":
        if GEN_COL not in df.columns:
            raise ValueError(f"{csv_path} is an NSGA-II population file but has no '{GEN_COL}' column.")

        for benchmark, sub in df.groupby("_benchmark"):
            max_gen = sub[GEN_COL].max()
            last_gen_rows = sub[sub[GEN_COL] == max_gen]

            for _, row in last_gen_rows.iterrows():
                attack = float(row[ATTACK_COL])
                distortion = float(row[DIST_COL])
                grouped_rows[str(benchmark)].append((-attack, distortion))

    else:
        for _, row in df.iterrows():
            benchmark = str(row["_benchmark"])
            attack = float(row[ATTACK_COL])
            distortion = float(row[DIST_COL])
            grouped_rows[benchmark].append((-attack, distortion))

    fronts: Dict[str, np.ndarray] = {}
    for benchmark, points in grouped_rows.items():
        fronts[benchmark] = nondominated_min(np.asarray(points, dtype=float))

    return fronts



def discover_front_records(
    data_root: Path,
    mapping: Dict[str, str],
    vlm_name: str,
) -> List[FrontRecord]:
    """
    Merge CSV fragments under one logical run.

    Group key:
      (vlm, run_id, approach, benchmark)

    Merge policy:
      - NSGA-II: prefer pareto fragments; else use population fragments
      - Random: merge all random fragments
    """
    csv_files = sorted(data_root.rglob("*.csv"), key=lambda p: natural_key(str(p)))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found under: {data_root}")

    grouped_fragments = defaultdict(lambda: defaultdict(list))
    grouped_paths = defaultdict(lambda: defaultdict(list))

    for csv_path in csv_files:
        approach = detect_approach(csv_path)
        if approach is None:
            continue

        run_id = extract_run_id(csv_path)
        fragment_kind = detect_fragment_kind(csv_path, approach)

        per_benchmark_fronts = load_csv_fragment_fronts(
            csv_path=csv_path,
            mapping=mapping,
            approach=approach,
        )

        for benchmark, front_min in per_benchmark_fronts.items():
            key = (vlm_name, run_id, approach, benchmark)
            grouped_fragments[key][fragment_kind].append(front_min)
            grouped_paths[key][fragment_kind].append(str(csv_path))

    records: List[FrontRecord] = []

    for key, kind_dict in grouped_fragments.items():
        vlm, run_id, approach, benchmark = key

        if approach == NSGA_LABEL:
            if "pareto" in kind_dict and len(kind_dict["pareto"]) > 0:
                chosen_fronts = kind_dict["pareto"]
                chosen_paths = grouped_paths[key]["pareto"]
            elif "population" in kind_dict and len(kind_dict["population"]) > 0:
                chosen_fronts = kind_dict["population"]
                chosen_paths = grouped_paths[key]["population"]
            else:
                merged = []
                merged_paths = []
                for frag_kind, fronts in kind_dict.items():
                    merged.extend(fronts)
                    merged_paths.extend(grouped_paths[key][frag_kind])
                chosen_fronts = merged
                chosen_paths = merged_paths
        else:
            chosen_fronts = []
            chosen_paths = []
            for frag_kind, fronts in kind_dict.items():
                chosen_fronts.extend(fronts)
                chosen_paths.extend(grouped_paths[key][frag_kind])

        union_points = np.vstack(chosen_fronts)
        merged_front = nondominated_min(union_points)

        records.append(
            FrontRecord(
                vlm=vlm,
                run_id=run_id,
                approach=approach,
                benchmark=benchmark,
                csv_paths=sorted(set(chosen_paths), key=natural_key),
                front_min=merged_front,
            )
        )

    if not records:
        raise RuntimeError(
            "No NSGA-II or Random CSV files were detected. "
            "Check file names and run folder structure."
        )

    return sorted(
        records,
        key=lambda r: (
            natural_key(r.vlm),
            natural_key(r.benchmark),
            natural_key(r.approach),
            natural_key(r.run_id),
        ),
    )


def filter_to_matched_pairs(records: List[FrontRecord]) -> Tuple[List[FrontRecord], pd.DataFrame]:
    """
    Keep only matched comparisons:
      same (vlm, benchmark, run_id) present in BOTH approaches.
    """
    by_key = defaultdict(dict)
    for rec in records:
        key = (rec.vlm, rec.benchmark, rec.run_id)
        by_key[key][rec.approach] = rec

    matched_records: List[FrontRecord] = []
    coverage_rows = []

    grouped = defaultdict(list)
    for (vlm, benchmark, run_id), apps in by_key.items():
        has_nsga = NSGA_LABEL in apps
        has_random = RANDOM_LABEL in apps

        coverage_rows.append({
            "vlm": vlm,
            "benchmark": benchmark,
            "run_id": run_id,
            "has_nsga": has_nsga,
            "has_random": has_random,
            "matched": has_nsga and has_random,
        })

        if has_nsga and has_random:
            matched_records.append(apps[NSGA_LABEL])
            matched_records.append(apps[RANDOM_LABEL])
            grouped[(vlm, benchmark)].append(run_id)

    coverage_df = pd.DataFrame(coverage_rows).sort_values(
        ["vlm", "benchmark", "run_id"],
        key=lambda s: s.map(natural_key) if s.dtype == object else s,
    ).reset_index(drop=True)

    return matched_records, coverage_df


def compute_empirical_reference_front(records: List[FrontRecord]) -> Dict[Tuple[str, str], np.ndarray]:
    pooled: Dict[Tuple[str, str], List[np.ndarray]] = defaultdict(list)

    for rec in records:
        pooled[(rec.vlm, rec.benchmark)].append(rec.front_min)

    reference_fronts = {}
    for key, fronts in pooled.items():
        union_points = np.vstack(fronts)
        reference_fronts[key] = nondominated_min(union_points)

    return reference_fronts


def compute_hv_reference_points(
    records: List[FrontRecord],
    margin_ratio: float = 0.05,
) -> Dict[Tuple[str, str], List[float]]:
    pooled: Dict[Tuple[str, str], List[np.ndarray]] = defaultdict(list)

    for rec in records:
        pooled[(rec.vlm, rec.benchmark)].append(rec.front_min)

    hv_reference_points = {}
    for key, fronts in pooled.items():
        all_points = np.vstack(fronts)
        nadir = np.max(all_points, axis=0)
        ideal = np.min(all_points, axis=0)
        span = nadir - ideal
        span = np.where(span <= 1e-12, 1.0, span)
        ref_point = nadir + margin_ratio * span
        hv_reference_points[key] = [float(ref_point[0]), float(ref_point[1])]

    return hv_reference_points


def compute_hv(front_min: np.ndarray, reference_point: List[float]) -> float:
    front_min = nondominated_min(front_min)
    metric = HyperVolume(reference_point=reference_point)

    try:
        return float(metric.compute(points_to_solutions(front_min)))
    except Exception:
        return float(metric.compute(front_min))


def compute_igd(front_min: np.ndarray, reference_front_min: np.ndarray) -> float:
    front_min = nondominated_min(front_min)
    reference_front_min = nondominated_min(reference_front_min)

    try:
        metric = InvertedGenerationalDistance(reference_front=points_to_solutions(reference_front_min))
        return float(metric.compute(points_to_solutions(front_min)))
    except Exception:
        try:
            metric = InvertedGenerationalDistance(reference_front_min)
            return float(metric.compute(front_min))
        except Exception:
            try:
                metric = InvertedGenerationalDistance(points_to_solutions(reference_front_min))
                return float(metric.compute(points_to_solutions(front_min)))
            except Exception:
                metric = InvertedGenerationalDistance(reference_front_min)
                return float(metric.compute(front_min))


def compute_run_level_metrics(records: List[FrontRecord]) -> Tuple[pd.DataFrame, Dict[Tuple[str, str], np.ndarray], Dict[Tuple[str, str], List[float]]]:
    reference_fronts = compute_empirical_reference_front(records)
    hv_reference_points = compute_hv_reference_points(records)

    rows = []
    for rec in records:
        key = (rec.vlm, rec.benchmark)
        ref_front = reference_fronts[key]
        hv_ref = hv_reference_points[key]

        hv = compute_hv(rec.front_min, hv_ref)
        igd = compute_igd(rec.front_min, ref_front)

        rows.append(
            {
                "vlm": rec.vlm,
                "benchmark": rec.benchmark,
                "run_id": rec.run_id,
                "approach": rec.approach,
                "csv_paths": " | ".join(rec.csv_paths),
                "n_points": int(len(rec.front_min)),
                "n_ref_points": int(len(ref_front)),
                "HV": float(hv),
                "IGD": float(igd),
                "HV_ref_f1": float(hv_ref[0]),
                "HV_ref_f2": float(hv_ref[1]),
            }
        )

    #--------Debugg--------------
    print("rows generated:", len(rows))

    if len(rows) > 0:
        print("row keys:", rows[0].keys())
    #----------------------------
    metrics_df = pd.DataFrame(rows).sort_values(
        by=["vlm", "benchmark", "approach", "run_id"],
        key=lambda s: s.map(natural_key) if s.dtype == object else s,
    ).reset_index(drop=True)

    return metrics_df, reference_fronts, hv_reference_points


def summarize_metrics(metrics_df: pd.DataFrame) -> pd.DataFrame:
    grouped = (
        metrics_df.groupby(["vlm", "benchmark", "approach"], as_index=False)
        .agg(
            n_runs=("run_id", "nunique"),
            hv_mean=("HV", "mean"),
            hv_std=("HV", "std"),
            igd_mean=("IGD", "mean"),
            igd_std=("IGD", "std"),
            n_points_mean=("n_points", "mean"),
            n_ref_points=("n_ref_points", "max"),
        )
        .sort_values(
            ["vlm", "benchmark", "approach"],
            key=lambda s: s.map(natural_key) if s.dtype == object else s,
        )
        .reset_index(drop=True)
    )
    return grouped


def summarize_coverage(coverage_df: pd.DataFrame) -> pd.DataFrame:
    return (
        coverage_df.groupby(["vlm", "benchmark"], as_index=False)
        .agg(
            n_runs_seen=("run_id", "nunique"),
            nsga_runs=("has_nsga", "sum"),
            random_runs=("has_random", "sum"),
            matched_runs=("matched", "sum"),
        )
        .sort_values(
            ["vlm", "benchmark"],
            key=lambda s: s.map(natural_key) if s.dtype == object else s,
        )
        .reset_index(drop=True)
    )


def set_plot_style():
    if HAS_SEABORN:
        sns.set_theme(style="whitegrid", context="paper", font_scale=1.0)
    plt.rcParams["figure.dpi"] = PLOT_DPI
    plt.rcParams["savefig.dpi"] = PLOT_DPI


def plot_run_distributions(metrics_df: pd.DataFrame, out_dir: Path) -> None:
    ensure_dir(out_dir)
    set_plot_style()

    for vlm, sub in metrics_df.groupby("vlm"):
        benchmarks = sorted(sub["benchmark"].unique(), key=natural_key)
        fig, axes = plt.subplots(2, 1, figsize=(max(10, 1.1 * len(benchmarks)), 9), sharex=True)

        for ax, metric, ylabel in zip(
            axes,
            ["HV", "IGD"],
            ["Hypervolume (higher is better)", "IGD (lower is better)"]
        ):
            plot_df = sub.copy()
            plot_df["benchmark"] = pd.Categorical(plot_df["benchmark"], categories=benchmarks, ordered=True)

            if HAS_SEABORN:
                sns.boxplot(data=plot_df, x="benchmark", y=metric, hue="approach", ax=ax)
                sns.stripplot(
                    data=plot_df,
                    x="benchmark",
                    y=metric,
                    hue="approach",
                    dodge=True,
                    alpha=0.65,
                    linewidth=0.5,
                    edgecolor="black",
                    ax=ax
                )
                handles, labels = ax.get_legend_handles_labels()
                ax.legend(handles[:2], labels[:2], title="Approach", loc="best")
            else:
                for i, benchmark in enumerate(benchmarks):
                    tmp = plot_df[plot_df["benchmark"] == benchmark]
                    nsga_vals = tmp[tmp["approach"] == NSGA_LABEL][metric].values
                    rand_vals = tmp[tmp["approach"] == RANDOM_LABEL][metric].values

                    if len(nsga_vals):
                        ax.boxplot(nsga_vals, positions=[i - 0.15], widths=0.25)
                    if len(rand_vals):
                        ax.boxplot(rand_vals, positions=[i + 0.15], widths=0.25)

                ax.set_xticks(range(len(benchmarks)))
                ax.set_xticklabels(benchmarks, rotation=45, ha="right")

            ax.set_title(f"{vlm}: {metric} across matched runs")
            ax.set_ylabel(ylabel)
            ax.tick_params(axis="x", rotation=45)

        axes[-1].set_xlabel("Benchmark")
        fig.tight_layout()
        fig.savefig(out_dir / f"{vlm}_rq1_run_distributions.png", bbox_inches="tight")
        plt.close(fig)


def plot_benchmark_means(summary_df: pd.DataFrame, out_dir: Path) -> None:
    ensure_dir(out_dir)
    set_plot_style()

    for vlm, sub in summary_df.groupby("vlm"):
        benchmarks = sorted(sub["benchmark"].unique(), key=natural_key)
        fig, axes = plt.subplots(1, 2, figsize=(14, max(6, 0.45 * len(benchmarks))), sharey=True)

        for ax, mean_col, title in zip(
            axes,
            ["hv_mean", "igd_mean"],
            ["Mean HV (higher is better)", "Mean IGD (lower is better)"]
        ):
            pivot = sub.pivot(index="benchmark", columns="approach", values=mean_col).reindex(benchmarks)

            y = np.arange(len(pivot.index))
            nsga_vals = pivot[NSGA_LABEL].values if NSGA_LABEL in pivot.columns else np.full(len(y), np.nan)
            rand_vals = pivot[RANDOM_LABEL].values if RANDOM_LABEL in pivot.columns else np.full(len(y), np.nan)

            for i, (a, b) in enumerate(zip(nsga_vals, rand_vals)):
                ax.plot([a, b], [i, i], linewidth=1.5, alpha=0.8)

            ax.scatter(nsga_vals, y, label=NSGA_LABEL, s=55)
            ax.scatter(rand_vals, y, label=RANDOM_LABEL, s=55)
            ax.set_yticks(y)
            ax.set_yticklabels(pivot.index)
            ax.set_xlabel(mean_col)
            ax.set_title(f"{vlm}: {title}")
            ax.legend(loc="best")

        fig.tight_layout()
        fig.savefig(out_dir / f"{vlm}_rq1_benchmark_means.png", bbox_inches="tight")
        plt.close(fig)


In [ ]:
# ============================================================================
# RQ1 Results for - BLIP
# ============================================================================

data_root = Path("/home/user_name/input/blip_output_v9/")
output_dir = Path("/home/user_name/output/")
mapping_csv = None
vlm_name = "BLIP"

ensure_dir(output_dir)
ensure_dir(output_dir / "plots")

mapping = load_mapping(mapping_csv)

all_records = discover_front_records(
    data_root=data_root,
    mapping=mapping,
    vlm_name=vlm_name,
)

matched_records, coverage_df = filter_to_matched_pairs(all_records)

metrics_df, reference_fronts, hv_reference_points = compute_run_level_metrics(matched_records)
summary_df = summarize_metrics(metrics_df)
coverage_summary_df = summarize_coverage(coverage_df)

run_level_path = output_dir / "run_level_metrics.csv"
summary_path = output_dir / "benchmark_summary.csv"
coverage_path = output_dir / "coverage_by_run.csv"
coverage_summary_path = output_dir / "coverage_summary.csv"
ref_sizes_path = output_dir / "reference_front_sizes.csv"

metrics_df.to_csv(run_level_path, index=False)
summary_df.to_csv(summary_path, index=False)
coverage_df.to_csv(coverage_path, index=False)
coverage_summary_df.to_csv(coverage_summary_path, index=False)

ref_rows = []
for (vlm, benchmark), ref_front in reference_fronts.items():
    ref_rows.append({
        "vlm": vlm,
        "benchmark": benchmark,
        "n_ref_points": len(ref_front),
        "hv_ref_f1": hv_reference_points[(vlm, benchmark)][0],
        "hv_ref_f2": hv_reference_points[(vlm, benchmark)][1],
    })
pd.DataFrame(ref_rows).to_csv(ref_sizes_path, index=False)

plot_run_distributions(metrics_df, output_dir / "plots")
plot_benchmark_means(summary_df, output_dir / "plots")

print(f"Saved: {run_level_path}")
print(f"Saved: {summary_path}")
print(f"Saved: {coverage_path}")
print(f"Saved: {coverage_summary_path}")
print(f"Saved: {ref_sizes_path}")

print("\nImportant note:")
print("This jMetalPy-only version computes HV and IGD correctly for matched runs.")
print("It does NOT compute Mann-Whitney U, A12, paired Wilcoxon, or Cohen's dz.")

print("\nCoverage summary:")
display(coverage_summary_df)

print("\nRun-level metrics:")
display(metrics_df.head(5))

print("\nBenchmark summary:")
display(summary_df.head(10))

## BLIP Statistical Analysis

In [ ]:
# ============================================================
# RQ1 statistics from existing run_level_metrics.csv - BLIP
# ============================================================
# Library-only implementation:
# - Pingouin: Mann-Whitney U, A12 via CLES, Wilcoxon, Cohen's dz

from pathlib import Path
import numpy as np
import pandas as pd
import pingouin as pg
from scipy.stats import mannwhitneyu
# ----------------------------
# Paths
# ----------------------------
input_csv = Path("/home/user_name/input/run_level_metrics.csv")
output_dir = Path("/home/user_name/input/output/output_stats")
output_dir.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Load and validate
# ----------------------------
df = pd.read_csv(input_csv)

required_cols = {"vlm", "benchmark", "run_id", "approach", "HV", "IGD"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"run_level_metrics.csv is missing columns: {sorted(missing)}")

df["vlm"] = df["vlm"].astype(str)
df["benchmark"] = df["benchmark"].astype(str)
df["run_id"] = df["run_id"].astype(str)
df["approach"] = df["approach"].astype(str)

NSGA_LABEL = "NSGA-II"
RANDOM_LABEL = "Random"

# Keep only rows from the two approaches of interest
df = df[df["approach"].isin([NSGA_LABEL, RANDOM_LABEL])].copy()

# ---------------------------------------------------------
# 1) Per-benchmark statistics over repeated runs
#    For each (vlm, benchmark):
#      - HV: Mann-Whitney U + A12(CLES)
#      - IGD: Mann-Whitney U + A12(CLES on -IGD)
# ---------------------------------------------------------
per_benchmark_rows = []

for (vlm, benchmark), sub in df.groupby(["vlm", "benchmark"], sort=False):
    nsga = sub[sub["approach"] == NSGA_LABEL].copy()
    rand = sub[sub["approach"] == RANDOM_LABEL].copy()

    if nsga.empty or rand.empty:
        continue

    hv_nsga = nsga["HV"].dropna().to_numpy()
    hv_rand = rand["HV"].dropna().to_numpy()

    igd_nsga = nsga["IGD"].dropna().to_numpy()
    igd_rand = rand["IGD"].dropna().to_numpy()

    """
    # HV: higher is better
    hv_stats = pg.mwu(hv_nsga, hv_rand, alternative="two-sided")
    # IGD: lower is better -> use negative values so "larger is better"
    igd_stats = pg.mwu(-igd_nsga, -igd_rand, alternative="two-sided")
    hv_u = float(hv_stats["U_val"].iloc[0])
    hv_p = float(hv_stats["p_val"].iloc[0])
    hv_a12 = float(hv_stats["CLES"].iloc[0])
    igd_u = float(igd_stats["U_val"].iloc[0])
    igd_p = float(igd_stats["p_val"].iloc[0])
    igd_a12 = float(igd_stats["CLES"].iloc[0])
    """
    # Updated Formula with Shaukats Paper
    # HV: higher is better
    hv_u, hv_p = mannwhitneyu(hv_nsga, hv_rand, alternative="two-sided")
    hv_a12 = hv_u / (len(hv_nsga) * len(hv_rand))

    # IGD: lower is better -> negate so A12 > 0.5 favors NSGA-II
    igd_u, igd_p = mannwhitneyu(-igd_nsga, -igd_rand, alternative="two-sided")
    igd_a12 = igd_u / (len(igd_nsga) * len(igd_rand))

    per_benchmark_rows.append({
        "vlm": vlm,
        "benchmark": benchmark,
        #"n_runs_nsga": len(hv_nsga),
        #"n_runs_random": len(hv_rand),

        "hv_mean_nsga": float(np.mean(hv_nsga)),
        "hv_mean_random": float(np.mean(hv_rand)),
        #"hv_median_nsga": float(np.median(hv_nsga)),
        #"hv_median_random": float(np.median(hv_rand)),
        "hv_mwu_u": hv_u,
        "hv_mwu_p": hv_p,
        "hv_Pval_better": ("Significant difference" if hv_p < 0.05 else "No difference"),
        "hv_a12": hv_a12,   # updated with new shaukat's formula
        #"hv_better": NSGA_LABEL if np.mean(hv_nsga) > np.mean(hv_rand) else RANDOM_LABEL,
        "hv_A12_better": NSGA_LABEL if hv_a12 > 0.5 else (RANDOM_LABEL if hv_a12 < 0.5 else "Tie"),


        "igd_mean_nsga": float(np.mean(igd_nsga)),
        "igd_mean_random": float(np.mean(igd_rand)),
        #"igd_median_nsga": float(np.median(igd_nsga)),
        #"igd_median_random": float(np.median(igd_rand)),
        #"igd_mwu_u": igd_u,
        #"igd_mwu_p": igd_p,
        #"igd_a12": igd_a12,  # with shaukats reference paper
        #"igd_better": NSGA_LABEL if np.mean(igd_nsga) < np.mean(igd_rand) else RANDOM_LABEL,
        "igd_better": NSGA_LABEL if igd_a12 > 0.5 else (RANDOM_LABEL if igd_a12 < 0.5 else "Tie"),
    })

per_benchmark_stats = pd.DataFrame(per_benchmark_rows).sort_values(
    ["vlm", "benchmark"]
).reset_index(drop=True)

# ---------------------------------------------------------
# 2) Benchmark-level means from repeated runs
#    One mean HV and mean IGD per (vlm, benchmark, approach)
# ---------------------------------------------------------
benchmark_means = (
    df.groupby(["vlm", "benchmark", "approach"], as_index=False)
      .agg(
          mean_hv=("HV", "mean"),
          mean_igd=("IGD", "mean"),
          n_runs=("run_id", "nunique"),
      )
)

# ---------------------------------------------------------
# 3) Across-benchmark paired statistics per VLM
#    For each VLM:
#      - pair benchmark means NSGA-II vs Random
#      - Wilcoxon signed-rank
#      - paired Cohen's dz
# ---------------------------------------------------------
across_rows = []

for vlm, sub in benchmark_means.groupby("vlm", sort=False):
    hv_pivot = sub.pivot(index="benchmark", columns="approach", values="mean_hv").dropna()
    igd_pivot = sub.pivot(index="benchmark", columns="approach", values="mean_igd").dropna()

    # HV
    if NSGA_LABEL in hv_pivot.columns and RANDOM_LABEL in hv_pivot.columns and len(hv_pivot) > 0:
        hv_nsga = hv_pivot[NSGA_LABEL].to_numpy()
        hv_rand = hv_pivot[RANDOM_LABEL].to_numpy()

        hv_w = pg.wilcoxon(hv_nsga, hv_rand, alternative="two-sided")
        hv_w_stat = float(hv_w["W_val"].iloc[0])
        hv_w_p = float(hv_w["p_val"].iloc[0])

        hv_dz = float(pg.compute_effsize(hv_nsga, hv_rand, paired=True, eftype="cohen"))
    else:
        hv_nsga = np.array([])
        hv_rand = np.array([])
        hv_w_stat = np.nan
        hv_w_p = np.nan
        hv_dz = np.nan

    # IGD
    if NSGA_LABEL in igd_pivot.columns and RANDOM_LABEL in igd_pivot.columns and len(igd_pivot) > 0:
        igd_nsga = igd_pivot[NSGA_LABEL].to_numpy()
        igd_rand = igd_pivot[RANDOM_LABEL].to_numpy()

        igd_w = pg.wilcoxon(igd_nsga, igd_rand, alternative="two-sided")
        igd_w_stat = float(igd_w["W_val"].iloc[0])
        igd_w_p = float(igd_w["p_val"].iloc[0])

        # lower IGD is better, so negate to make positive dz favor NSGA-II
        igd_dz = float(pg.compute_effsize(-igd_nsga, -igd_rand, paired=True, eftype="cohen"))
    else:
        igd_nsga = np.array([])
        igd_rand = np.array([])
        igd_w_stat = np.nan
        igd_w_p = np.nan
        igd_dz = np.nan

    across_rows.append({
        "vlm": vlm,
        "n_benchmarks_hv": int(len(hv_pivot)) if 'hv_pivot' in locals() else 0,
        "n_benchmarks_igd": int(len(igd_pivot)) if 'igd_pivot' in locals() else 0,

        "hv_mean_nsga": float(np.mean(hv_nsga)) if len(hv_nsga) else np.nan,
        "hv_mean_random": float(np.mean(hv_rand)) if len(hv_rand) else np.nan,
        "hv_wilcoxon_w": hv_w_stat,
        "hv_wilcoxon_p": hv_w_p,
        "hv_cohens_dz": hv_dz,

        "igd_mean_nsga": float(np.mean(igd_nsga)) if len(igd_nsga) else np.nan,
        "igd_mean_random": float(np.mean(igd_rand)) if len(igd_rand) else np.nan,
        "igd_wilcoxon_w": igd_w_stat,
        "igd_wilcoxon_p": igd_w_p,
        "igd_cohens_dz": igd_dz,
    })

across_benchmarks_stats = pd.DataFrame(across_rows).sort_values("vlm").reset_index(drop=True)

# ---------------------------------------------------------
# 4) Optional: compact table for paper use
# ---------------------------------------------------------
rq1_table_hv = per_benchmark_stats[[
    "vlm", "benchmark",
    "hv_mean_nsga", "hv_mean_random", "hv_mwu_u",
    "hv_mwu_p", "hv_Pval_better", "hv_a12", "hv_A12_better"
]].copy()

rq1_table_igd = per_benchmark_stats[[
    "vlm", "benchmark",
    "igd_mean_nsga", "igd_mean_random", "igd_better"
    #"igd_mwu_u", "igd_mwu_p", "igd_a12", "igd_better"
]].copy()

# ---------------------------------------------------------
# 5) Save
# ---------------------------------------------------------
per_benchmark_path = output_dir / "rq1_per_benchmark_stats.csv"
benchmark_means_path = output_dir / "rq1_benchmark_mean_metrics.csv"
across_path = output_dir / "rq1_across_benchmarks_stats.csv"
hv_table_path = output_dir / "rq1_table_hv.csv"
#igd_table_path = output_dir / "rq1_table_igd.csv"

per_benchmark_stats.to_csv(per_benchmark_path, index=False)
benchmark_means.to_csv(benchmark_means_path, index=False)
across_benchmarks_stats.to_csv(across_path, index=False)
rq1_table_hv.to_csv(hv_table_path, index=False)
#rq1_table_igd.to_csv(igd_table_path, index=False)

print(f"Saved: {per_benchmark_path}")
print(f"Saved: {benchmark_means_path}")
print(f"Saved: {across_path}")
print(f"Saved: {hv_table_path}")
#print(f"Saved: {igd_table_path}")

display(per_benchmark_stats.head(30))
display(across_benchmarks_stats)

In [ ]:
# ============================================================================
# RQ1 Results for - CLIP
# ============================================================================

data_root = Path("/home/user_name/input/")
output_dir = Path("/home/user_name/output")
mapping_csv = None
vlm_name = "CLIP"

ensure_dir(output_dir)
ensure_dir(output_dir / "plots")

mapping = load_mapping(mapping_csv)

all_records = discover_front_records(
    data_root=data_root,
    mapping=mapping,
    vlm_name=vlm_name,
)

matched_records, coverage_df = filter_to_matched_pairs(all_records)

metrics_df, reference_fronts, hv_reference_points = compute_run_level_metrics(matched_records)
summary_df = summarize_metrics(metrics_df)
coverage_summary_df = summarize_coverage(coverage_df)

run_level_path = output_dir / "run_level_metrics.csv"
summary_path = output_dir / "benchmark_summary.csv"
coverage_path = output_dir / "coverage_by_run.csv"
coverage_summary_path = output_dir / "coverage_summary.csv"
ref_sizes_path = output_dir / "reference_front_sizes.csv"

metrics_df.to_csv(run_level_path, index=False)
summary_df.to_csv(summary_path, index=False)
coverage_df.to_csv(coverage_path, index=False)
coverage_summary_df.to_csv(coverage_summary_path, index=False)

ref_rows = []
for (vlm, benchmark), ref_front in reference_fronts.items():
    ref_rows.append({
        "vlm": vlm,
        "benchmark": benchmark,
        "n_ref_points": len(ref_front),
        "hv_ref_f1": hv_reference_points[(vlm, benchmark)][0],
        "hv_ref_f2": hv_reference_points[(vlm, benchmark)][1],
    })
pd.DataFrame(ref_rows).to_csv(ref_sizes_path, index=False)

plot_run_distributions(metrics_df, output_dir / "plots")
plot_benchmark_means(summary_df, output_dir / "plots")

print(f"Saved: {run_level_path}")
print(f"Saved: {summary_path}")
print(f"Saved: {coverage_path}")
print(f"Saved: {coverage_summary_path}")
print(f"Saved: {ref_sizes_path}")

print("\nImportant note:")
print("This jMetalPy-only version computes HV and IGD correctly for matched runs.")
print("It does NOT compute Mann-Whitney U, A12, paired Wilcoxon, or Cohen's dz.")

print("\nCoverage summary:")
display(coverage_summary_df)

print("\nRun-level metrics:")
display(metrics_df.head(5))

print("\nBenchmark summary:")
display(summary_df.head(10))

## CLIP Statistical Analysis

In [ ]:
# ============================================================
# RQ1 statistics from existing run_level_metrics.csv - DeepSeek
# ============================================================
# Library-only implementation:
# - Pingouin: Mann-Whitney U, A12 via CLES, Wilcoxon, Cohen's dz


from pathlib import Path
import numpy as np
import pandas as pd
import pingouin as pg
from scipy.stats import mannwhitneyu
# ----------------------------
# Paths
# ----------------------------
input_csv = Path("/home/user_name/input//run_level_metrics.csv")
output_dir = Path("/home/user_name/onput/")
output_dir.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Load and validate
# ----------------------------
df = pd.read_csv(input_csv)

required_cols = {"vlm", "benchmark", "run_id", "approach", "HV", "IGD"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"run_level_metrics.csv is missing columns: {sorted(missing)}")

df["vlm"] = df["vlm"].astype(str)
df["benchmark"] = df["benchmark"].astype(str)
df["run_id"] = df["run_id"].astype(str)
df["approach"] = df["approach"].astype(str)

NSGA_LABEL = "NSGA-II"
RANDOM_LABEL = "Random"

# Keep only rows from the two approaches of interest
df = df[df["approach"].isin([NSGA_LABEL, RANDOM_LABEL])].copy()

# ---------------------------------------------------------
# 1) Per-benchmark statistics over repeated runs
#    For each (vlm, benchmark):
#      - HV: Mann-Whitney U + A12(CLES)
#      - IGD: Mann-Whitney U + A12(CLES on -IGD)
# ---------------------------------------------------------
per_benchmark_rows = []

for (vlm, benchmark), sub in df.groupby(["vlm", "benchmark"], sort=False):
    nsga = sub[sub["approach"] == NSGA_LABEL].copy()
    rand = sub[sub["approach"] == RANDOM_LABEL].copy()

    if nsga.empty or rand.empty:
        continue

    hv_nsga = nsga["HV"].dropna().to_numpy()
    hv_rand = rand["HV"].dropna().to_numpy()

    igd_nsga = nsga["IGD"].dropna().to_numpy()
    igd_rand = rand["IGD"].dropna().to_numpy()


    # Updated Formula with Shaukats Paper
    # HV: higher is better
    hv_u, hv_p = mannwhitneyu(hv_nsga, hv_rand, alternative="two-sided")
    hv_a12 = hv_u / (len(hv_nsga) * len(hv_rand))

    # IGD: lower is better -> negate so A12 > 0.5 favors NSGA-II
    igd_u, igd_p = mannwhitneyu(-igd_nsga, -igd_rand, alternative="two-sided")
    igd_a12 = igd_u / (len(igd_nsga) * len(igd_rand))

    per_benchmark_rows.append({
        "vlm": vlm,
        "benchmark": benchmark,
        #"n_runs_nsga": len(hv_nsga),
        #"n_runs_random": len(hv_rand),

        "hv_mean_nsga": float(np.mean(hv_nsga)),
        "hv_mean_random": float(np.mean(hv_rand)),
        #"hv_median_nsga": float(np.median(hv_nsga)),
        #"hv_median_random": float(np.median(hv_rand)),
        "hv_mwu_u": hv_u,
        "hv_mwu_p": hv_p,
        "hv_Pval_better": ("Significant difference" if hv_p < 0.05 else "No difference"),
        "hv_a12": hv_a12,   # updated with new shaukat's formula
        #"hv_better": NSGA_LABEL if np.mean(hv_nsga) > np.mean(hv_rand) else RANDOM_LABEL,
        "hv_A12_better": NSGA_LABEL if hv_a12 > 0.5 else (RANDOM_LABEL if hv_a12 < 0.5 else "Tie"),


        "igd_mean_nsga": float(np.mean(igd_nsga)),
        "igd_mean_random": float(np.mean(igd_rand)),
        "igd_better": NSGA_LABEL if igd_a12 > 0.5 else (RANDOM_LABEL if igd_a12 < 0.5 else "Tie"),
    })

per_benchmark_stats = pd.DataFrame(per_benchmark_rows).sort_values(
    ["vlm", "benchmark"]
).reset_index(drop=True)

# ---------------------------------------------------------
# 2) Benchmark-level means from repeated runs
#    One mean HV and mean IGD per (vlm, benchmark, approach)
# ---------------------------------------------------------
benchmark_means = (
    df.groupby(["vlm", "benchmark", "approach"], as_index=False)
      .agg(
          mean_hv=("HV", "mean"),
          mean_igd=("IGD", "mean"),
          n_runs=("run_id", "nunique"),
      )
)

across_rows = []

for vlm, sub in benchmark_means.groupby("vlm", sort=False):
    hv_pivot = sub.pivot(index="benchmark", columns="approach", values="mean_hv").dropna()
    igd_pivot = sub.pivot(index="benchmark", columns="approach", values="mean_igd").dropna()

    # HV
    if NSGA_LABEL in hv_pivot.columns and RANDOM_LABEL in hv_pivot.columns and len(hv_pivot) > 0:
        hv_nsga = hv_pivot[NSGA_LABEL].to_numpy()
        hv_rand = hv_pivot[RANDOM_LABEL].to_numpy()

        hv_w = pg.wilcoxon(hv_nsga, hv_rand, alternative="two-sided")
        hv_w_stat = float(hv_w["W_val"].iloc[0])
        hv_w_p = float(hv_w["p_val"].iloc[0])

        hv_dz = float(pg.compute_effsize(hv_nsga, hv_rand, paired=True, eftype="cohen"))
    else:
        hv_nsga = np.array([])
        hv_rand = np.array([])
        hv_w_stat = np.nan
        hv_w_p = np.nan
        hv_dz = np.nan

    # IGD
    if NSGA_LABEL in igd_pivot.columns and RANDOM_LABEL in igd_pivot.columns and len(igd_pivot) > 0:
        igd_nsga = igd_pivot[NSGA_LABEL].to_numpy()
        igd_rand = igd_pivot[RANDOM_LABEL].to_numpy()

        igd_w = pg.wilcoxon(igd_nsga, igd_rand, alternative="two-sided")
        igd_w_stat = float(igd_w["W_val"].iloc[0])
        igd_w_p = float(igd_w["p_val"].iloc[0])

        # lower IGD is better, so negate to make positive dz favor NSGA-II
        igd_dz = float(pg.compute_effsize(-igd_nsga, -igd_rand, paired=True, eftype="cohen"))
    else:
        igd_nsga = np.array([])
        igd_rand = np.array([])
        igd_w_stat = np.nan
        igd_w_p = np.nan
        igd_dz = np.nan

    across_rows.append({
        "vlm": vlm,
        "n_benchmarks_hv": int(len(hv_pivot)) if 'hv_pivot' in locals() else 0,
        "n_benchmarks_igd": int(len(igd_pivot)) if 'igd_pivot' in locals() else 0,

        "hv_mean_nsga": float(np.mean(hv_nsga)) if len(hv_nsga) else np.nan,
        "hv_mean_random": float(np.mean(hv_rand)) if len(hv_rand) else np.nan,
        "hv_wilcoxon_w": hv_w_stat,
        "hv_wilcoxon_p": hv_w_p,
        "hv_cohens_dz": hv_dz,

        "igd_mean_nsga": float(np.mean(igd_nsga)) if len(igd_nsga) else np.nan,
        "igd_mean_random": float(np.mean(igd_rand)) if len(igd_rand) else np.nan,
        "igd_wilcoxon_w": igd_w_stat,
        "igd_wilcoxon_p": igd_w_p,
        "igd_cohens_dz": igd_dz,
    })

across_benchmarks_stats = pd.DataFrame(across_rows).sort_values("vlm").reset_index(drop=True)

# ---------------------------------------------------------
# 4) Optional: compact table for paper use
# ---------------------------------------------------------
rq1_table_hv = per_benchmark_stats[[
    "vlm", "benchmark",
    "hv_mean_nsga", "hv_mean_random", "hv_mwu_u",
    "hv_mwu_p", "hv_Pval_better", "hv_a12", "hv_A12_better"
]].copy()

rq1_table_igd = per_benchmark_stats[[
    "vlm", "benchmark",
    "igd_mean_nsga", "igd_mean_random", "igd_better"
    #"igd_mwu_u", "igd_mwu_p", "igd_a12", "igd_better"
]].copy()

# ---------------------------------------------------------
# 5) Save
# ---------------------------------------------------------
per_benchmark_path = output_dir / "rq1_per_benchmark_stats.csv"
benchmark_means_path = output_dir / "rq1_benchmark_mean_metrics.csv"
across_path = output_dir / "rq1_across_benchmarks_stats.csv"
hv_table_path = output_dir / "rq1_table_hv.csv"
#igd_table_path = output_dir / "rq1_table_igd.csv"

per_benchmark_stats.to_csv(per_benchmark_path, index=False)
benchmark_means.to_csv(benchmark_means_path, index=False)
across_benchmarks_stats.to_csv(across_path, index=False)
rq1_table_hv.to_csv(hv_table_path, index=False)
#rq1_table_igd.to_csv(igd_table_path, index=False)

print(f"Saved: {per_benchmark_path}")
print(f"Saved: {benchmark_means_path}")
print(f"Saved: {across_path}")
print(f"Saved: {hv_table_path}")
#print(f"Saved: {igd_table_path}")

display(per_benchmark_stats.head(30))
display(across_benchmarks_stats)

####################################################
## RQ1 - Large Benchmark Table
###################################################

In [ ]:
# ============================================================
# RQ1 paper visualization script
# Builds:
#   1) One large 30-row summary table for CLIP and BLIP
#   2) Two boxplots (CLIP and BLIP), comparing NSGA-II vs Random
#      using benchmark-level mean metric values, annotated with
#      overall mean A12 and overall mean p-value
#   3) Overleaf-ready LaTeX code for:
#      - large summary table
#      - overall means table
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

# ------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------
CLIP_STATS_CSV = Path("/home/user_name/input/rq1_per_benchmark_stats.csv") # clip
BLIP_STATS_CSV = Path("/home/user_name/input/rq1_per_benchmark_stats.csv") # blip

OUTPUT_DIR = Path("/home/simula/workspace/paper2/rq1_paper_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METRIC = "hv"   # "hv" or "igd"
ALPHA = 0.05
PLOT_DPI = 300

# ------------------------------------------------------------
# STYLE
# ------------------------------------------------------------
def set_plot_style():
    if HAS_SEABORN:
        sns.set_theme(style="whitegrid", context="paper", font_scale=1.0)
    plt.rcParams["figure.dpi"] = PLOT_DPI
    plt.rcParams["savefig.dpi"] = PLOT_DPI

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def natural_key(text: str):
    import re
    return [int(tok) if tok.isdigit() else tok.lower() for tok in re.split(r"(\d+)", str(text))]

def build_benchmark_label_map(clip_df: pd.DataFrame, blip_df: pd.DataFrame) -> dict:
    all_benchmarks = sorted(
        set(clip_df["benchmark"].astype(str)).union(set(blip_df["benchmark"].astype(str))),
        key=natural_key
    )
    return {bench: f"B{i:02d}" for i, bench in enumerate(all_benchmarks, start=1)}

def validate_columns(df: pd.DataFrame, metric: str, vlm_name: str):
    needed = {
        "benchmark",
        f"{metric}_mean_nsga",
        f"{metric}_mean_random",
        f"{metric}_mwu_p",
        f"{metric}_a12",
    }
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"{vlm_name} CSV is missing columns: {sorted(missing)}")

def a12_label(a12_value: float) -> str:
    if pd.isna(a12_value):
        return "Error!"

    strength = max(float(a12_value), 1.0 - float(a12_value))

    if strength >= 0.71:
        return "High"
    elif strength >= 0.64:
        return "Medium"
    elif strength >= 0.56:
        return "Low"
    else:
        return "Low"

def best_algo_from_stats(a12_value: float, p_value: float, alpha: float = 0.05) -> str:
    if pd.isna(a12_value) or pd.isna(p_value):
        return "Error!"
    if float(p_value) >= alpha:
        return "No Sign."
    if float(a12_value) > 0.5:
        return "NSGA-II"
    elif float(a12_value) < 0.5:
        return "RAND"
    else:
        return "Error!"

def prepare_vlm_table(df: pd.DataFrame, vlm_prefix: str, metric: str) -> pd.DataFrame:
    validate_columns(df, metric, vlm_prefix)

    out = df.copy()
    out["benchmark"] = out["benchmark"].astype(str)

    a12_col = f"{metric}_a12"
    p_col = f"{metric}_mwu_p"

    out[f"{vlm_prefix}_A12_Label"] = out[a12_col].apply(a12_label)
    out[f"{vlm_prefix}_p_value"] = out[p_col].astype(float).round(6)
    out[f"{vlm_prefix}_Best"] = [
        best_algo_from_stats(a12, p, alpha=ALPHA)
        for a12, p in zip(out[a12_col], out[p_col])
    ]

    return out[[
        "benchmark",
        f"{vlm_prefix}_A12_Label",
        f"{vlm_prefix}_p_value",
        f"{vlm_prefix}_Best",
    ]].copy()

def make_combined_table(clip_df: pd.DataFrame, blip_df: pd.DataFrame, metric: str) -> pd.DataFrame:
    clip_tab = prepare_vlm_table(clip_df, "CLIP", metric)
    blip_tab = prepare_vlm_table(blip_df, "BLIP", metric)

    merged = pd.merge(
        clip_tab,
        blip_tab,
        on="benchmark",
        how="outer"
    )

    merged = merged.sort_values("benchmark", key=lambda s: s.map(natural_key)).reset_index(drop=True)
    merged = merged.fillna("N/A")

    final_table = merged[[
        "benchmark",
        "CLIP_p_value", "CLIP_A12_Label", "CLIP_Best" ,
        "BLIP_p_value", "BLIP_A12_Label",  "BLIP_Best",
    ]].copy()

    return final_table

def compute_overall_means(df: pd.DataFrame, metric: str) -> dict:
    return {
        "mean_a12": float(df[f"{metric}_a12"].mean()),
        "mean_p": float(df[f"{metric}_mwu_p"].mean()),
        "mean_nsga": float(df[f"{metric}_mean_nsga"].mean()),
        "mean_random": float(df[f"{metric}_mean_random"].mean()),
    }

def build_boxplot_dataframe(df: pd.DataFrame, vlm_name: str, metric: str) -> pd.DataFrame:
    return pd.DataFrame({
        "VLM": [vlm_name] * (2 * len(df)),
        "Algorithm": ["NSGA-II"] * len(df) + ["Random"] * len(df),
        "Metric_Value": list(df[f"{metric}_mean_nsga"].astype(float)) + list(df[f"{metric}_mean_random"].astype(float)),
    })

def plot_vlm_boxplot(df: pd.DataFrame, vlm_name: str, metric: str, out_path: Path):
    stats = compute_overall_means(df, metric)
    plot_df = build_boxplot_dataframe(df, vlm_name, metric)

    plt.figure(figsize=(6, 5))

    if HAS_SEABORN:
        sns.boxplot(data=plot_df, x="Algorithm", y="Metric_Value")
        sns.stripplot(data=plot_df, x="Algorithm", y="Metric_Value", color="black", alpha=0.6, size=4)
    else:
        nsga_vals = plot_df[plot_df["Algorithm"] == "NSGA-II"]["Metric_Value"].to_numpy()
        rand_vals = plot_df[plot_df["Algorithm"] == "Random"]["Metric_Value"].to_numpy()
        plt.boxplot([nsga_vals, rand_vals], labels=["NSGA-II", "Random"])

    ylabel = "Benchmark-level mean HV" if metric == "hv" else "Benchmark-level mean IGD"
    title_metric = "HV" if metric == "hv" else "IGD"

    plt.ylabel(ylabel)
    plt.title(
        f"{vlm_name}: {title_metric} comparison\n"
        f"Mean A12 = {stats['mean_a12']:.3f}, Mean p-value = {stats['mean_p']:.3f}"
    )
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()

def render_table_as_figure(table_df: pd.DataFrame, out_path: Path, title: str):
    n_rows = len(table_df)
    fig_h = max(8, 0.35 * (n_rows + 2))
    fig_w = 12

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis("off")

    tbl = ax.table(
        cellText=table_df.values,
        colLabels=table_df.columns,
        cellLoc="center",
        loc="center"
    )

    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    tbl.scale(1, 1.2)

    plt.title(title, fontsize=12, pad=20)
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()

# ------------------------------------------------------------
# LATEX HELPERS
# ------------------------------------------------------------
def latex_escape(text):
    text = str(text)
    replacements = {
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
    }
    for k, v in replacements.items():
        text = text.replace(k, v)
    return text

def write_large_table_latex(table_df: pd.DataFrame, out_path: Path, caption: str, label: str):
    lines = []
    lines.append(r"\begin{table*}[tb]")
    lines.append(r"\centering")
    lines.append(r"\caption{" + caption + "}")
    lines.append(r"\label{" + label + "}")
    lines.append(r"\resizebox{\textwidth}{!}{%")
    lines.append(r"\begin{tabular}{lccc|ccc}")
    lines.append(r"\toprule")
    lines.append(r" & \multicolumn{3}{c|}{CLIP} & \multicolumn{3}{c}{BLIP} \\")
    lines.append(r"\cmidrule(lr){2-4} \cmidrule(lr){5-7}")
    lines.append(r"Benchmark & p-value & A12 & Best Approach & p-value & A12 & Best Approach \\")
    lines.append(r"\midrule")

    for _, row in table_df.iterrows():
        bench = latex_escape(row["benchmark"])
        c_a12 = latex_escape(row["CLIP_A12_Label"])
        c_p   = latex_escape(row["CLIP_p_value"])
        c_b   = latex_escape(row["CLIP_Best"])
        b_a12 = latex_escape(row["BLIP_A12_Label"])
        b_p   = latex_escape(row["BLIP_p_value"])
        b_b   = latex_escape(row["BLIP_Best"])

        lines.append(f"{bench} & {c_p} & {c_a12} & {c_b} & {b_p} & {b_a12} & {b_b} \\\\")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}%")
    lines.append(r"}")
    lines.append(r"\end{table*}")

    out_path.write_text("\n".join(lines), encoding="utf-8")

def write_overall_means_latex(overall_df: pd.DataFrame, out_path: Path, caption: str, label: str):
    lines = []
    lines.append(r"\begin{table}[tb]")
    lines.append(r"\centering")
    lines.append(r"\caption{" + caption + "}")
    lines.append(r"\label{" + label + "}")
    lines.append(r"\begin{tabular}{lcccc}")
    lines.append(r"\toprule")
    lines.append(r"VLM & Mean A12 & Mean p-value & Mean NSGA-II & Mean Random \\")
    lines.append(r"\midrule")

    for _, row in overall_df.iterrows():
        vlm = latex_escape(row["VLM"])
        a12 = f"{row['Mean_A12']:.4f}"
        p   = f"{row['Mean_p_value']:.4f}"
        ng  = f"{row['Mean_NSGA']:.4f}"
        rd  = f"{row['Mean_Random']:.4f}"
        lines.append(f"{vlm} & {a12} & {p} & {ng} & {rd} \\\\")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")

    out_path.write_text("\n".join(lines), encoding="utf-8")

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------
set_plot_style()

clip_df = pd.read_csv(CLIP_STATS_CSV)
blip_df = pd.read_csv(BLIP_STATS_CSV)

clip_df["benchmark"] = clip_df["benchmark"].astype(str)
blip_df["benchmark"] = blip_df["benchmark"].astype(str)

benchmark_label_map = build_benchmark_label_map(clip_df, blip_df)

clip_df["benchmark_raw"] = clip_df["benchmark"]
blip_df["benchmark_raw"] = blip_df["benchmark"]

clip_df["benchmark"] = clip_df["benchmark"].map(benchmark_label_map)
blip_df["benchmark"] = blip_df["benchmark"].map(benchmark_label_map)

validate_columns(clip_df, METRIC, "DSeek")
validate_columns(blip_df, METRIC, "BLIP")

# ------------------------------------------------------------
# TABLE
# ------------------------------------------------------------
combined_table = make_combined_table(clip_df, blip_df, METRIC)

table_csv = OUTPUT_DIR / f"rq1_combined_table_{METRIC}.csv"
table_xlsx = OUTPUT_DIR / f"rq1_combined_table_{METRIC}.xlsx"
table_png = OUTPUT_DIR / f"rq1_combined_table_{METRIC}.png"
table_tex = OUTPUT_DIR / f"rq1_combined_table_{METRIC}.tex"

combined_table.to_csv(table_csv, index=False)
combined_table.to_excel(table_xlsx, index=False)

render_table_as_figure(
    combined_table,
    table_png,
    title=f"RQ1 Summary Table ({METRIC.upper()}-based comparison)"
)

write_large_table_latex(
    combined_table,
    table_tex,
    caption=f"RQ1 - summary based on Hypervolume (HV) comparison where A12 effect size is reported as Low, Medium or High. Best approach shows NSGA-II, random search (RAND), and No Sign. for no significant difference.",
    label=f"tab:rq1_summary_{METRIC}"
)

# ------------------------------------------------------------
# OVERALL MEANS
# ------------------------------------------------------------

clip_means = compute_overall_means(clip_df, METRIC)
blip_means = compute_overall_means(blip_df, METRIC)

overall_means_df = pd.DataFrame([
    {
        "VLM": "CLIP",
        "Mean_A12": clip_means["mean_a12"],
        "Mean_p_value": clip_means["mean_p"],
        "Mean_NSGA": clip_means["mean_nsga"],
        "Mean_Random": clip_means["mean_random"],
    },
    {
        "VLM": "BLIP",
        "Mean_A12": blip_means["mean_a12"],
        "Mean_p_value": blip_means["mean_p"],
        "Mean_NSGA": blip_means["mean_nsga"],
        "Mean_Random": blip_means["mean_random"],
    },
])

overall_means_csv = OUTPUT_DIR / f"rq1_overall_means_{METRIC}.csv"
overall_means_tex = OUTPUT_DIR / f"rq1_overall_means_{METRIC}.tex"
overall_means_df.to_csv(overall_means_csv, index=False)

write_overall_means_latex(
    overall_means_df,
    overall_means_tex,
    caption=f"Overall mean A12, p-value, and benchmark-level mean {METRIC.upper()} values used to summarize RQ1.",
    label=f"tab:rq1_overall_means_{METRIC}"
)


benchmark_map_df = pd.DataFrame({
    "benchmark_raw": list(benchmark_label_map.keys()),
    "benchmark_paper": list(benchmark_label_map.values())
})
benchmark_map_df.to_csv(OUTPUT_DIR / "benchmark_name_mapping.csv", index=False)
# ------------------------------------------------------------
# BOXPLOTS
# ------------------------------------------------------------
clip_boxplot = OUTPUT_DIR / f"DSeek_{METRIC}_boxplot.png"
blip_boxplot = OUTPUT_DIR / f"BLIP_{METRIC}_boxplot.png"

plot_vlm_boxplot(clip_df, "CLIP", METRIC, clip_boxplot)
plot_vlm_boxplot(blip_df, "BLIP", METRIC, blip_boxplot)

# ------------------------------------------------------------
# DONE
# ------------------------------------------------------------
print(f"Saved: {table_csv}")
print(f"Saved: {table_xlsx}")
print(f"Saved: {table_png}")
print(f"Saved: {table_tex}")
print(f"Saved: {overall_means_csv}")
print(f"Saved: {overall_means_tex}")
print(f"Saved: {clip_boxplot}")
print(f"Saved: {blip_boxplot}")

display(combined_table.head(30))
#display(overall_means_df)

In [ ]:
######################################
#               END - RQ1            #
######################################

## RQ2

In [ ]:
################################################
#      RQ2 - Split Part(A) and Part B)        #
###############################################

In [ ]:
import pandas as pd

def generate_rq2_split_tables(results_dict, vlm_type="BLIP"):
    """
    Splits the analysis of metamorphic testing results into:
      - RQ2 (a): Structural Exploration (Search Space & Frequencies)
      - RQ2 (b): Violations & Defect Impact (Testing Lethality)
    """
    records = []
    total_attempts_by_approach = {'NSGAII': 0, 'Random': 0}

    # 1. Flatten the processed nested metrics dictionary
    for approach, combos in results_dict.items():
        for combo, counts in combos.items():
            violations = counts.get('violations', 0)
            attempts = counts.get('total attempts', 0)

            total_attempts_by_approach[approach] += attempts
            records.append({
                'Combination': combo,
                'Approach': approach,
                'violations': violations,
                'total attempts': attempts,
                'Selected MRs Count': len(combo.split('+')) if combo else 0
            })

    df_base = pd.DataFrame(records)
    if df_base.empty:
        print("Error: No data available for execution.")
        return

    t_nsga = total_attempts_by_approach['NSGAII']
    t_rand = total_attempts_by_approach['Random']

    # -------------------------------------------------------------------------
    # PART 1: RQ2 (a) - Number of MRs and Combinations (Search Layer)
    # -------------------------------------------------------------------------
    # Pivot to compare the search allocation profiles of both approaches side-by-side
    df_a = df_base.pivot(
        index=['Combination', 'Selected MRs Count'],
        columns='Approach',
        values='total attempts'
    ).fillna(0).reset_index()

    # Structural handling in case one approach is missing entries
    for app in ['NSGAII', 'Random']:
        if app not in df_a.columns:
            df_a[app] = 0.0

    df_a['NSGA-II Selection %'] = (df_a['NSGAII'] / t_nsga * 100) if t_nsga > 0 else 0.0
    df_a['Random Selection %'] = (df_a['Random'] / t_rand * 100) if t_rand > 0 else 0.0

    df_a = df_a.rename(columns={'NSGAII': 'NSGA-II Attempts', 'Random': 'Random Attempts'})

    # Sort by the configurations most explored by your evolutionary search
    top_10_a = df_a.sort_values(by='NSGA-II Attempts', ascending=False).head(10)

    # Console Output (RQ2a)
    print("\n==============================================================================")
    print(f"      RQ2 (a): STRUCTURAL EXPLORATION TABLE (Top 10) --- {vlm_type}           ")
    print("==============================================================================")
    print(top_10_a.to_string(index=False, formatters={
        'NSGA-II Selection %': '{:.2f}%'.format,
        'Random Selection %': '{:.2f}%'.format,
        'NSGA-II Attempts': '{:,.0f}'.format,
        'Random Attempts': '{:,.0f}'.format
    }))

    # LaTeX Output (RQ2a)
    print(f"\n% --- Overleaf Ready LaTeX Code Block for RQ2 (a) ---")
    latex_a = top_10_a.copy()
    latex_a['Combination'] = latex_a['Combination'].str.replace('_', '\\_')
    latex_a['NSGA-II Selection \\%'] = latex_a['NSGA-II Selection %'].map('{:.2f}\\%'.format)
    latex_a['Random Selection \\%'] = latex_a['Random Selection %'].map('{:.2f}\\%'.format)
    latex_a = latex_a.drop(columns=['NSGA-II Selection %', 'Random Selection %'])
    latex_a.columns = [col.replace('_', '\\_') for col in latex_a.columns]

    print(latex_a.to_latex(
        index=False,
        column_format="clrrrr",
        caption=f"RQ2(a): Combinatorial Exploration Patterns of Subsumed Spaces for {vlm_type}",
        label=f"tab:rq2a_{vlm_type.lower()}",
        position="th"
    ))

    # -------------------------------------------------------------------------
    # PART 2: RQ2 (b) - Violations of these MRs (Impact Layer)
    # -------------------------------------------------------------------------
    df_base['violation_rate'] = df_base['violations'] / df_base['total attempts']

    # Reorder layout metrics matching the naming schema parameters
    column_order_b = ['Combination', 'Selected MRs Count', 'Approach', 'violations', 'total attempts', 'violation_rate']
    df_b = df_base[column_order_b]
    top_10_b = df_b.sort_values(by='violations', ascending=False).head(10)

    # Console Output (RQ2b)
    print("\n==============================================================================")
    print(f"      RQ2 (b): MUTATION VIOLATION IMPACT TABLE (Top 10) --- {vlm_type}         ")
    print("==============================================================================")
    print(top_10_b.to_string(index=False, formatters={'violation_rate': '{:.6f}'.format}))

    # LaTeX Output (RQ2b)
    print(f"\n% --- Overleaf Ready LaTeX Code Block for RQ2 (b) ---")
    latex_b = top_10_b.copy()
    latex_b['Combination'] = latex_b['Combination'].str.replace('_', '\\_')
    latex_b['violation\\_rate'] = latex_b['violation_rate'].map('{:.6f}'.format)
    latex_b = latex_b.drop(columns=['violation_rate'])
    latex_b.columns = [col.replace('_', '\\_').title() for col in latex_b.columns]

    print(latex_b.to_latex(
        index=False,
        column_format="clrrrc",
        caption=f"RQ2(b): Metamorphic Violation and Destructive Failure Performance for {vlm_type}",
        label=f"tab:rq2b_{vlm_type.lower()}",
        position="th"
    ))

# ------------------------------------------------------------
# EXECUTION
# ------------------------------------------------------------
# Pass the 'output' dictionary generated from your previous data execution run:
generate_rq2_split_tables(output, "BLIP")

##########################################
## RQ2.1: Updated with CLIP thresholds
########################################

In [ ]:
import pandas as pd
import json
import ast
import itertools
from pathlib import Path
import numpy as np

# Tracking 6 MRs to form the structural patterns
MMR_COLS = ['MMR1', 'MMR2', 'MMR3', 'MMR4', 'MMR5', 'MMR6']
TARGET_LABELS = ['object', 'trash', 'animal', 'vegetation']
CLIP_THRESHOLDS = [0.01, 0.03, 0.05]

def parse_vlm_output(val_str):
    """Safely extracts dictionary configurations or fallback scalar values."""
    if pd.isna(val_str): return None
    s = str(val_str).strip()
    try:
        cleaned = s.replace('{', '').replace('}', '').replace("'", "").replace('"', '')
        if ':' in cleaned:
            d = {}
            for p in cleaned.split(','):
                if ':' not in p: continue
                k, v = p.split(':')
                d[k.strip()] = float(v.strip())
            return d
    except:
        pass
    try:
        return float(s)
    except:
        return None

def preprocess_df(df, is_random):
    """Maps approach-specific columns to standard boolean flags for all MRs."""
    actual_cols = {}
    for target in MMR_COLS:
        found = [c for c in df.columns if target in c]
        if found:
            actual_cols[target] = found[0]

    if is_random and 'Selected_MMRs' in df.columns:
        for target in MMR_COLS:
            col_name = actual_cols.get(target, target)
            df[col_name] = df['Selected_MMRs'].apply(lambda x: target in str(x))
            actual_cols[target] = col_name

    return df, actual_cols

def build_complete_base_space():
    all_combos = []
    for r in range(1, 7):
        for combo in itertools.combinations(['M1', 'M2', 'M3', 'M4', 'M5', 'M6'], r):
            all_combos.append("+".join(combo))
    return all_combos

def extract_vlm_order_metrics(root_path, vlm_name):
    """Extracts row-level execution attempts and multi-threshold violations."""
    base_combinations = build_complete_base_space()
    records = []

    path_obj = Path(root_path)
    if not path_obj.exists():
        return []

    for csv_file in path_obj.rglob("*.csv"):
        if any(x in csv_file.name.lower() for x in ["summary", "metrics", "report", "stats"]):
            continue

        is_random = "outputRand" in str(csv_file) or "random" in str(csv_file).lower()
        approach = 'Random' if is_random else 'VLMTest'

        try:
            df = pd.read_csv(csv_file)
            df, actual_cols = preprocess_df(df, is_random)

            orig_col, trans_col = None, None
            for col in df.columns:
                c_low = col.lower()
                if 'orig' in c_low and ('cat' in c_low or 'conf' in c_low): orig_col = col
                if 'trans' in c_low and ('cat' in c_low or 'conf' in c_low): trans_col = col

            if not orig_col or not trans_col: continue

            active_mrs = [target for target in MMR_COLS if target in actual_cols]
            df['combination'] = df.apply(
                lambda row: "+".join([f"M{c[-1]}" for c in active_mrs if row[actual_cols[c]]]), axis=1
            )

            for orig_raw, trans_raw, combo in zip(df[orig_col], df[trans_col], df['combination']):
                if combo == "" or combo not in base_combinations: continue
                order = len(combo.split('+'))

                orig_val = parse_vlm_output(orig_raw)
                trans_val = parse_vlm_output(trans_raw)

                if orig_val is None or trans_val is None: continue

                record = {
                    'VLM': vlm_name,
                    'Approach': approach,
                    'Order': order
                }

                # BLIP Evaluation Logic
                if vlm_name == "BLIP":
                    if isinstance(orig_val, dict) and isinstance(trans_val, dict):
                        orig_total = sum(orig_val.values())
                        trans_total = sum(trans_val.values())
                        record['BLIP_Viol'] = 1 if abs(orig_total - trans_total) >= 1 else 0
                    else:
                        record['BLIP_Viol'] = 0

                # CLIP Evaluation Logic across 5 discrete thresholds
                elif vlm_name == "CLIP":
                    if isinstance(orig_val, dict) and isinstance(trans_val, dict):
                        shifts = {}
                        for label in TARGET_LABELS:
                            orig_conf = sum([v for k, v in orig_val.items() if label in str(k).lower()])
                            trans_conf = sum([v for k, v in trans_val.items() if label in str(k).lower()])
                            shifts[label] = abs(orig_conf - trans_conf)

                        for t in CLIP_THRESHOLDS:
                            record[f'CLIP_Viol_{t}'] = 1 if any(s >= t for s in shifts.values()) else 0
                    elif isinstance(orig_val, (int, float)) and isinstance(trans_val, (int, float)):
                        diff = abs(orig_val - trans_val)
                        for t in CLIP_THRESHOLDS:
                            record[f'CLIP_Viol_{t}'] = 1 if diff >= t else 0

                records.append(record)
        except Exception:
            continue

    return records

def run_rq2_1_analysis(clip_path, blip_path):
    all_records = []
    all_records.extend(extract_vlm_order_metrics(clip_path, "CLIP"))
    all_records.extend(extract_vlm_order_metrics(blip_path, "BLIP"))

    df_master = pd.DataFrame(all_records)
    if df_master.empty:
        print("Error: No data compiled from target experiment tracks.")
        return

    # Calculate approach budgets dynamically to handle execution splits
    approach_budgets = df_master.groupby(['VLM', 'Approach']).size().to_dict()

    # =========================================================================
    # PROCESSING SECTION 1: BLIP HORIZONTAL WIDE MATRIX
    # =========================================================================
    df_blip_raw = df_master[df_master['VLM'] == 'BLIP']
    if not df_blip_raw.empty:
        blip_grouped = df_blip_raw.groupby(['Order', 'Approach']).agg(
            Attempts=('VLM', 'count'),
            Violations=('BLIP_Viol', 'sum')
        ).reset_index()

        blip_grouped['Selection %'] = blip_grouped.apply(
            lambda r: (r['Attempts'] / approach_budgets[('BLIP', r['Approach'])] * 100), axis=1
        )
        blip_grouped['Violation Rate'] = blip_grouped['Violations'] / blip_grouped['Attempts']

        # Horizontal Wide Pivot Transformation
        blip_pivot = blip_grouped.pivot(index='Order', columns='Approach', values=['Selection %', 'Violation Rate']).fillna(0.0)

        blip_wide = pd.DataFrame({'Order': blip_pivot.index})
        for app in ['VLMTest', 'Random']:
            blip_wide[f'{app} Generation Frequency'] = blip_pivot[('Selection %', app)].values
            blip_wide[f'{app} Violation Rate'] = blip_pivot[('Violation Rate', app)].values

        print("\n" + "="*115)
        print("      RQ2.1: OVERALL BLIP PATTERN GENERATION & METAMORPHIC VIOLATION SUMMARY BY COMPLEXITY ORDER        ")
        print("="*115)
        print(blip_wide.to_string(index=False, formatters={
            'VLMTest Generation Frequency': '{:.2f}%'.format, 'VLMTest Violation Rate': lambda x: f"{x*100:.2f}%",
            'Random Generation Frequency': '{:.2f}%'.format, 'Random Violation Rate': lambda x: f"{x*100:.2f}%"
        }))

        # Hand-crafted multi-column toprule LaTeX Generator
        print("\n% --- LaTeX Code for BLIP Wide Matrix Table ---")
        print("\\begin{table}[th]\n\\centering\n\\caption{RQ2.1: BLIP Horizontal Configuration Matrix}\n\\label{tab:rq2_1_blip_wide}")
        print("\\begin{tabular}{crrrr}\n\\toprule\n& \\multicolumn{2}{c}{VLMTest} & \\multicolumn{2}{c}{Random} \\\\")
        print("\\cmidrule(lr){2-3} \\cmidrule(lr){4-5}\nOrder & Gen Freq & Violation Rate & Gen Freq & Violation Rate \\\\\n\\midrule")
        for _, r in blip_wide.iterrows():
            print(f"{int(r['Order'])} & {r['VLMTest Generation Frequency']:.1f}\\ & {r['VLMTest Violation Rate']*100:.1f}\\ & {r['Random Generation Frequency']:.1f}\\ & {r['Random Violation Rate']*100:.1f}\\ \\\\")
        print("\\bottomrule\n\\end{tabular}\n\\end{table}")

    # =========================================================================
    # PROCESSING SECTION 2: CLIP HORIZONTAL WIDE MATRIX (MULTI-THRESHOLD)
    # =========================================================================
    df_clip_raw = df_master[df_master['VLM'] == 'CLIP']
    if not df_clip_raw.empty:
        agg_dict = {'VLM': 'count'}
        for t in CLIP_THRESHOLDS:
            agg_dict[f'CLIP_Viol_{t}'] = 'sum'

        clip_grouped = df_clip_raw.groupby(['Order', 'Approach']).agg(agg_dict).reset_index()
        clip_grouped.rename(columns={'VLM': 'Attempts'}, inplace=True)

        clip_grouped['Selection %'] = clip_grouped.apply(
            lambda r: (r['Attempts'] / approach_budgets[('CLIP', r['Approach'])] * 100), axis=1
        )

        for t in CLIP_THRESHOLDS:
            clip_grouped[f'V-Rate ({t})'] = clip_grouped[f'CLIP_Viol_{t}'] / clip_grouped['Attempts']

        # Horizontal Wide Pivot Transformation
        val_cols = ['Selection %'] + [f'V-Rate ({t})' for t in CLIP_THRESHOLDS]
        clip_pivot = clip_grouped.pivot(index='Order', columns='Approach', values=val_cols).fillna(0.0)

        clip_wide = pd.DataFrame({'Order': clip_pivot.index})
        for app in ['VLMTest', 'Random']:
            clip_wide[f'{app} Generation Frequency'] = clip_pivot[('Selection %', app)].values
            for t in CLIP_THRESHOLDS:
                clip_wide[f'{app} V-Rate ({t})'] = clip_pivot[(f'V-Rate ({t})', app)].values

        print("\n" + "="*175)
        print("                                            RQ2.1: OVERALL CLIP PATTERN GENERATION & MULTI-THRESHOLD VIOLATION WIDE SUMMARY                                            ")
        print("="*175)

        fmt_dict = {}
        for app in ['VLMTest', 'Random']:
            fmt_dict[f'{app} Generation Frequency'] = '{:.2f}%'.format
            for t in CLIP_THRESHOLDS:
                fmt_dict[f'{app} V-Rate ({t})'] = lambda x: f"{x*100:.2f}%"

        #print(clip_wide.to_string(index=False, formatters=fmt_dict))

        # Hand-crafted multi-column toprule LaTeX Generator
        print("\n% --- LaTeX Code for CLIP Wide Multi-Threshold Matrix Table ---")
        print("\\begin{table}[th]\n\\centering\n\\caption{RQ2.1: CLIP Multi-Threshold Horizontal Performance Breakdown Across Complexity Orders}\n\\label{tab:rq2_1_clip_wide}")
        print("\\begin{tabular}{crrrrrrrrrrrr}\n\\toprule\n& \\multicolumn{6}{c}{VLMTest} & \\multicolumn{6}{c}{Random} \\\\")
        print("\\cmidrule(lr){2-7} \\cmidrule(lr){8-13}")
        print("Order & Freq & VR(0.01) & VR(0.03) & VR(0.05) & Freq & VR(0.01) & VR(0.03) & VR(0.05) \\\\\n\\midrule")
        for _, r in clip_wide.iterrows():
            line = f"{int(r['Order'])} & "
            line += f"{r['VLMTest Generation Frequency']:.1f}\\ & " + " & ".join([f"{r[f'VLMTest V-Rate ({t})']*100:.1f}\\" for t in CLIP_THRESHOLDS]) + " & "
            line += f"{r['Random Generation Frequency']:.1f}\\ & " + " & ".join([f"{r[f'Random V-Rate ({t})']*100:.1f}\\" for t in CLIP_THRESHOLDS]) + " \\\\"
            print(line)
        print("\\bottomrule\n\\end{tabular}\n\\end{table}")

if __name__ == "__main__":
    CLIP_DIR = "/home/user_name/input/"
    BLIP_DIR = "/home/user_name/input/"

    run_rq2_1_analysis(CLIP_DIR, BLIP_DIR)

In [ ]:
#############################3
# RQ2: CLIP with 3 thresholds - updated
#############################

## RQ2: part(b) (updated CLIP violation logic)

In [ ]:
import pandas as pd
import json
import ast
import itertools
from pathlib import Path

MMR_COLS = ['MMR1', 'MMR2', 'MMR3', 'MMR4', 'MMR5', 'MMR6']
TARGET_LABELS = ['object', 'trash', 'animal', 'vegetation']
CLIP_THRESHOLDS = [0.01, 0.03, 0.05]

def parse_vlm_output(val_str):
    """
    Advanced adaptive parser that handles integer counts, decimal confidence
    dictionaries, and single numeric float/integer values cleanly.
    """
    if pd.isna(val_str): return None
    s = str(val_str).strip()

    try:
        cleaned = s.replace('{', '').replace('}', '').replace("'", "").replace('"', '')
        if ':' in cleaned:
            d = {}
            for p in cleaned.split(','):
                if ':' not in p: continue
                k, v = p.split(':')
                d[k.strip()] = float(v.strip())
            return d
    except:
        pass

    try:
        return float(s)
    except:
        return None

def preprocess_df(df, is_random):
    """Maps approach-specific columns to standard boolean flags for all MRs."""
    actual_cols = {}
    for target in MMR_COLS:
        found = [c for c in df.columns if target in c]
        if found:
            actual_cols[target] = found[0]

    if is_random and 'Selected_MMRs' in df.columns:
        for target in MMR_COLS:
            col_name = actual_cols.get(target, target)
            df[col_name] = df['Selected_MMRs'].apply(lambda x: target in str(x))
            actual_cols[target] = col_name

    return df, actual_cols

def build_complete_base_space():
    all_combos = []
    for r in range(1, 7):
        for combo in itertools.combinations(['M1', 'M2', 'M3', 'M4', 'M5', 'M6'], r):
            all_combos.append("+".join(combo))
    return all_combos

def extract_vlm_violations(root_path, vlm_name):
    base_combinations = build_complete_base_space()
    storage = {'NSGAII': {}, 'Random': {}}

    # Initialize metric dictionaries dynamically based on VLM type
    for app in storage.keys():
        for combo in base_combinations:
            if vlm_name == "BLIP":
                storage[app][combo] = {'attempts': 0, 'violations': 0}
            else:
                storage[app][combo] = {'attempts': 0}
                for t in CLIP_THRESHOLDS:
                    storage[app][combo][f'violations_{t}'] = 0

    path_obj = Path(root_path)
    if not path_obj.exists():
        print(f"Warning: Directory path not found: {root_path}")
        return []

    for csv_file in path_obj.rglob("*.csv"):
        if any(x in csv_file.name.lower() for x in ["summary", "metrics", "report", "stats"]):
            continue

        is_random = "outputRand" in str(csv_file) or "random" in str(csv_file).lower()
        approach = 'Random' if is_random else 'NSGAII'

        try:
            df = pd.read_csv(csv_file)
            df, actual_cols = preprocess_df(df, is_random)

            orig_col, trans_col = None, None
            for col in df.columns:
                c_low = col.lower()
                if 'orig' in c_low and ('cat' in c_low or 'conf' in c_low): orig_col = col
                if 'trans' in c_low and ('cat' in c_low or 'conf' in c_low): trans_col = col

            if not orig_col or not trans_col: continue

            active_mrs = [target for target in MMR_COLS if target in actual_cols]
            df['combination'] = df.apply(
                lambda row: "+".join([f"M{c[-1]}" for c in active_mrs if row[actual_cols[c]]]), axis=1
            )

            for combo, orig_raw, trans_raw in zip(df['combination'], df[orig_col], df[trans_col]):
                if combo == "" or combo not in storage[approach]: continue

                orig_val = parse_vlm_output(orig_raw)
                trans_val = parse_vlm_output(trans_raw)
                if orig_val is None or trans_val is None: continue

                storage[approach][combo]['attempts'] += 1

                # --- BLIP Evaluation Rule Block ---
                if vlm_name == "BLIP":
                    if isinstance(orig_val, dict) and isinstance(trans_val, dict):
                        is_int_counts = all(v.is_integer() for v in orig_val.values()) and all(v.is_integer() for v in trans_val.values())
                        if is_int_counts and abs(sum(orig_val.values()) - sum(trans_val.values())) >= 1:
                            storage[approach][combo]['violations'] += 1

                # --- CLIP Multi-Threshold Rule Block ---
                elif vlm_name == "CLIP":
                    if isinstance(orig_val, dict) and isinstance(trans_val, dict):
                        shifts = {}
                        for label in TARGET_LABELS:
                            orig_conf = sum([v for k, v in orig_val.items() if label in str(k).lower()])
                            trans_conf = sum([v for k, v in trans_val.items() if label in str(k).lower()])
                            shifts[label] = abs(orig_conf - trans_conf)

                        for t in CLIP_THRESHOLDS:
                            if any(s >= t for s in shifts.values()):
                                storage[approach][combo][f'violations_{t}'] += 1

                    elif isinstance(orig_val, (int, float)) and isinstance(trans_val, (int, float)):
                        diff = abs(orig_val - trans_val)
                        for t in CLIP_THRESHOLDS:
                            if diff >= t:
                                storage[approach][combo][f'violations_{t}'] += 1
        except Exception:
            continue

    records = []
    for approach, combos in storage.items():
        for combo, metrics in combos.items():
            rec = {
                'VLM': vlm_name, 'Approach': approach, 'Combination': combo,
                'Order': len(combo.split('+')), 'Attempts': metrics['attempts']
            }
            if vlm_name == "BLIP":
                rec['Violations'] = metrics['violations']
            else:
                for t in CLIP_THRESHOLDS:
                    rec[f'Violations_{t}'] = metrics[f'violations_{t}']
            records.append(rec)

    return records

def run_rq2_2_analysis(clip_path, blip_path):
    all_records_clip = extract_vlm_violations(clip_path, "CLIP")
    all_records_blip = extract_vlm_violations(blip_path, "BLIP")

    # =========================================================================
    # PRODUCTION OUTPUT SECTION 1: BLIP SEPARATE TOP-10 SUMMARY
    # =========================================================================
    if all_records_blip:
        df_blip = pd.DataFrame(all_records_blip)

        # --- EXACT FIX: Compute total violations across all MR combinations ---
        total_nsga_viols = df_blip[df_blip['Approach'] == 'NSGAII']['Violations'].sum()
        total_rand_viols = df_blip[df_blip['Approach'] == 'Random']['Violations'].sum()

        nsga_top10 = df_blip[df_blip['Approach'] == 'NSGAII'].sort_values(by='Violations', ascending=False).head(10).reset_index(drop=True)
        rand_top10 = df_blip[df_blip['Approach'] == 'Random'].sort_values(by='Violations', ascending=False).head(10).reset_index(drop=True)
        # ------------------------------------------------------------------------

        # Calculate percentage contribution for each top pattern
        nsga_pct = (nsga_top10['Violations'] / total_nsga_viols * 100) if total_nsga_viols > 0 else 0.0
        rand_pct = (rand_top10['Violations'] / total_rand_viols * 100) if total_rand_viols > 0 else 0.0

        combined_blip = pd.DataFrame({
            'NSGA-II Combination': nsga_top10['Combination'],
            'NSGA-II Frequency': nsga_pct,
            'Random Combination': rand_top10['Combination'],
            'Random Frequency': rand_pct
        })

        # --- EXACT FIX: Map raw strings to enclosed numeric sets for BLIP console ---
        for col in ['NSGA-II Combination', 'Random Combination']:
            combined_blip[col] = combined_blip[col].astype(str).apply(
                lambda x: f"{{{x.replace('M', '').replace('+', ',')}}}" if x else ""
            )

        print("\n" + "="*95)
        print("    RQ2.2: COMBINED TOP-10 MOST DESTRUCTIVE PATTERNS SIDE-BY-SIDE --- BLIP    ")
        print("="*95)
        # Format the output frequencies to 1 decimal place
        print(combined_blip.to_string(index=False, formatters={'NSGA-II Frequency': '{:.1f}%'.format, 'Random Frequency': '{:.1f}%'.format}))

        print("\n% --- LaTeX Code for Combined Top-10 Table (BLIP) ---")
        # --- EXACT FIX: Escape curly braces to render as literals in BLIP LaTeX ---
        for _, r in combined_blip.iterrows():
            nsga_combo = str(r['NSGA-II Combination']).replace('{', '\\{').replace('}', '\\}')
            rand_combo = str(r['Random Combination']).replace('{', '\\{').replace('}', '\\}')
            print(f"{nsga_combo} & {r['NSGA-II Frequency']:.1f}\\% & {rand_combo} & {r['Random Frequency']:.1f}\\% \\\\")
        # ------------

        latex_blip = combined_blip.copy()
        latex_blip['NSGA-II Combination'] = latex_blip['NSGA-II Combination'].str.replace('_', '\\_')
        latex_blip['Random Combination'] = latex_blip['Random Combination'].str.replace('_', '\\_')
        print(latex_blip.to_latex(index=False, column_format="lrlr", position="th",
                                  caption="RQ2.2: Side-by-Side Comparison of Top-10 Failure Patterns for BLIP",
                                  label="tab:rq2_2_combined_blip"))

    # =========================================================================
    # PRODUCTION OUTPUT SECTION 2: CLIP COMPREHENSIVE HORIZONTAL WIDE MATRIX
    # =========================================================================
    if all_records_clip:
        df_clip = pd.DataFrame(all_records_clip)
        clip_horizontal_data = {}

        # Format string layout parameters across thresholds sequentially
        fmt_clip_dict = {}

        for t in CLIP_THRESHOLDS:
            viol_col = f'Violations_{t}'

            total_nsga_viols = df_clip[df_clip['Approach'] == 'NSGAII'][viol_col].sum()
            total_rand_viols = df_clip[df_clip['Approach'] == 'Random'][viol_col].sum()

            nsga_clip_top10 = df_clip[df_clip['Approach'] == 'NSGAII'].sort_values(by=viol_col, ascending=False).head(10).reset_index(drop=True)
            rand_clip_top10 = df_clip[df_clip['Approach'] == 'Random'].sort_values(by=viol_col, ascending=False).head(10).reset_index(drop=True)

            nsga_pct = (nsga_clip_top10[viol_col] / total_nsga_viols * 100) if total_nsga_viols > 0 else 0.0
            rand_pct = (rand_clip_top10[viol_col] / total_rand_viols * 100) if total_rand_viols > 0 else 0.0

            # === EXACT CODE FIX: Format directly during reindexing ===
            nsga_combos = nsga_clip_top10['Combination'].reindex(range(10), fill_value="").apply(
                lambda x: f"{{{x.replace('M', '').replace('+', ',')}}}" if x else ""
            )
            rand_combos = rand_clip_top10['Combination'].reindex(range(10), fill_value="").apply(
                lambda x: f"{{{x.replace('M', '').replace('+', ',')}}}" if x else ""
            )
            # =========================================================

            nsga_freqs = pd.Series(nsga_pct).reindex(range(10), fill_value=0.0)
            rand_freqs = pd.Series(rand_pct).reindex(range(10), fill_value=0.0)

            clip_horizontal_data[f'VLMTest Combo ({t})'] = nsga_combos
            clip_horizontal_data[f'VLMTest Freq ({t})'] = nsga_freqs
            clip_horizontal_data[f'Rand Combo ({t})'] = rand_combos
            clip_horizontal_data[f'Rand Freq ({t})'] = rand_freqs

            fmt_clip_dict[f'VLMTest Freq ({t})'] = '{:.1f}%'.format
            fmt_clip_dict[f'Rand Freq ({t})'] = '{:.1f}%'.format

        df_clip_wide = pd.DataFrame(clip_horizontal_data)
        # EXACT FIX: Strips 'M' and swaps '+' with ',' across all threshold column vectors
        #for t in CLIP_THRESHOLDS:
        #    for prefix in ['VLMTest', 'Rand']:
        #        col = f'{prefix} Combo ({t})'
        #        df_clip_wide[col] = df_clip_wide[col].astype(str).str.replace('M', '', regex=False).str.replace('+', ',', regex=False)

        print("\n" + "="*240)
        print("                                                         RQ2.2: CLIP COMPREHENSIVE MULTI-THRESHOLD HORIZONTAL PERFORMANCEbreakdown (ALL GATES SIDE-BY-SIDE)                                                         ")
        print("="*240)

        pd.set_option('display.width', 1000)
        print(df_clip_wide.to_string(index=False, formatters=fmt_clip_dict))

        # Hand-Crafted Academic Multi-Tier LaTeX Document Fragment Generator
        print("\n% --- LaTeX Code for Combined CLIP Horizontal Table ---")
        print("\\begin{table}[th]\n\\centering\n\\caption{RQ2.2: Horizontal Comparison of Top-10 Failure Patterns for CLIP Across Multi-Significance Thresholds}\n\\label{tab:rq2_2_clip_horizontal_wide}")

        # Build layout definition format containing 20 separate operational data tracks
        print("\\begin{tabular}{" + "lrlr"*len(CLIP_THRESHOLDS) + "}\n\\toprule")

        # Tier 1: Generate Master Multi-Column Headers Mapping Threshold Gates
        top_row = ""
        for i, t in enumerate(CLIP_THRESHOLDS):
            top_row += f"\\multicolumn{{4}}{{c}}{{Threshold {t}}}"
            if i < len(CLIP_THRESHOLDS) - 1: top_row += " & "
        print(top_row + " \\\\")

        # Tier 2: Draw Independent Split Mid-rules
        mid_rules = ""
        for i in range(len(CLIP_THRESHOLDS)):
            start_idx = 1 + (i * 4)
            end_idx = start_idx + 3
            mid_rules += f"\\cmidrule(lr){{{start_idx}-{end_idx}}}"
        print(mid_rules)

        # Tier 3: Output Column Subheadings
        # EXACT FIX: Update subheadings to represent Frequency distributions
        sub_headers = " & ".join(["MRs Combo & Freq (\\%) & MRs Combo & Freq (\\%)" for _ in CLIP_THRESHOLDS]) + " \\\\\n\\midrule"
        print(sub_headers)

        # Update LaTeX string cell engine loop to fetch floats and truncate to 1 decimal place
        # --- EXACT FIX: Escape curly braces to render as literals in LaTeX ---
        # === EXACT CODE FIX: Escape the curly braces for LaTeX compilation ===
        for idx in range(10):
            cells = []
            for t in CLIP_THRESHOLDS:
                v_combo = str(df_clip_wide.loc[idx, f'VLMTest Combo ({t})']).replace('{', '\\{').replace('}', '\\}')
                v_freq = float(df_clip_wide.loc[idx, f'VLMTest Freq ({t})'])
                r_combo = str(df_clip_wide.loc[idx, f'Rand Combo ({t})']).replace('{', '\\{').replace('}', '\\}')
                r_freq = float(df_clip_wide.loc[idx, f'Rand Freq ({t})'])
                cells.append(f"{v_combo} & {v_freq:.1f}\\% & {r_combo} & {r_freq:.1f}\\%")
            print(" & ".join(cells) + " \\\\")
        # ======================================================================

        print("\\bottomrule\n\\end{tabular}\n\\end{table}")

if __name__ == "__main__":
    CLIP_DIR = "/home/user_name/input/"
    BLIP_DIR = "/home/user_name/input/"

    run_rq2_2_analysis(CLIP_DIR, BLIP_DIR)

## RQ3

#############################################################
## RQ3: Mean Violation Strength | Updated CLIP 3  Thresholds
############################################################

In [ ]:
import pandas as pd
import json
import ast
import itertools
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import mannwhitneyu, rankdata

# Tracking 6 MRs to form the structural patterns (2^6 = 64 combinations)
MMR_COLS = ['MMR1', 'MMR2', 'MMR3', 'MMR4', 'MMR5', 'MMR6']
TARGET_LABELS = ['object', 'trash', 'animal', 'vegetation']
CLIP_THRESHOLDS = [0.01, 0.03, 0.05]

def parse_vlm_output(val_str):
    """Safely extracts dictionary configurations or fallback scalar values."""
    if pd.isna(val_str): return None
    s = str(val_str).strip()
    try:
        cleaned = s.replace('{', '').replace('}', '').replace("'", "").replace('"', '')
        if ':' in cleaned:
            d = {}
            for p in cleaned.split(','):
                if ':' not in p: continue
                k, v = p.split(':')
                d[k.strip()] = float(v.strip())
            return d
    except:
        pass
    try:
        return float(s)
    except:
        return None

def preprocess_df(df, is_random):
    """Maps approach-specific columns to standard boolean flags for all MRs."""
    actual_cols = {}
    for target in MMR_COLS:
        found = [c for c in df.columns if target in c]
        if found:
            actual_cols[target] = found[0]

    if is_random and 'Selected_MMRs' in df.columns:
        for target in MMR_COLS:
            col_name = actual_cols.get(target, target)
            df[col_name] = df['Selected_MMRs'].apply(lambda x: target in str(x))
            actual_cols[target] = col_name

    return df, actual_cols

def build_complete_base_space():
    all_combos = []
    for r in range(1, 7):
        for combo in itertools.combinations(['M1', 'M2', 'M3', 'M4', 'M5', 'M6'], r):
            all_combos.append("+".join(combo))
    return all_combos

def calculate_a12(seq1, seq2):
    """Computes the Vargha-Delaney A12 effect size statistic for independent groups."""
    n1 = len(seq1)
    n2 = len(seq2)
    if n1 == 0 or n2 == 0:
        return 0.5
    all_elements = np.concatenate([seq1, seq2])
    ranks = rankdata(all_elements)
    r1 = np.sum(ranks[:n1])
    a12 = (r1 / n1 - (n1 + 1) / 2) / n2
    return float(a12)

def extract_vlm_severity_records(root_path, vlm_name):
    """Processes directories to accumulate cascading metrics across multi-threshold tiers."""
    base_combinations = build_complete_base_space()
    records = []

    path_obj = Path(root_path)
    if not path_obj.exists():
        return []

    for csv_file in path_obj.rglob("*.csv"):
        if any(x in csv_file.name.lower() for x in ["summary", "metrics", "report", "stats"]):
            continue

        is_random = "outputRand" in str(csv_file) or "random" in str(csv_file).lower()
        approach = 'Random' if is_random else 'NSGAII'

        try:
            df = pd.read_csv(csv_file)
            df, actual_cols = preprocess_df(df, is_random)

            orig_col, trans_col = None, None
            for col in df.columns:
                c_low = col.lower()
                if 'orig' in c_low and ('cat' in c_low or 'conf' in c_low): orig_col = col
                if 'trans' in c_low and ('cat' in c_low or 'conf' in c_low): trans_col = col

            if not orig_col or not trans_col: continue

            bench_col = None
            for col in df.columns:
                cleaned_col_name = str(col).strip().lower()
                if 'bench' in cleaned_col_name or 'imag' in cleaned_col_name or 'scen' in cleaned_col_name:
                    bench_col = col
                    break
            if bench_col is None:
                df['extracted_bench'] = csv_file.stem.split('_')[0]
                bench_col = 'extracted_bench'

            active_mrs = [target for target in MMR_COLS if target in actual_cols]
            df['combination'] = df.apply(
                lambda row: "+".join([f"M{c[-1]}" for c in active_mrs if row[actual_cols[c]]]), axis=1
            )

            for orig_raw, trans_raw, combo, bench in zip(df[orig_col], df[trans_col], df['combination'], df[bench_col]):
                if combo == "" or combo not in base_combinations: continue
                order = len(combo.split('+'))

                orig_val = parse_vlm_output(orig_raw)
                trans_val = parse_vlm_output(trans_raw)
                if orig_val is None or trans_val is None: continue

                if vlm_name == "BLIP":
                    if isinstance(orig_val, dict) and isinstance(trans_val, dict):
                        orig_total = sum(orig_val.values())
                        trans_total = sum(trans_val.values())
                        if abs(orig_total - trans_total) >= 1:
                            records.append({
                                'VLM': 'BLIP', 'Approach': approach, 'Benchmark': str(bench).strip(),
                                'Order': order, 'Severity': float(abs(orig_total - trans_total))
                            })

                elif vlm_name == "CLIP":
                    rec = {
                        'VLM': 'CLIP', 'Approach': approach, 'Benchmark': str(bench).strip(), 'Order': order
                    }
                    has_any_violation = False

                    if isinstance(orig_val, dict) and isinstance(trans_val, dict):
                        shifts = {}
                        for label in TARGET_LABELS:
                            orig_conf = sum([v for k, v in orig_val.items() if label in str(k).lower()])
                            trans_conf = sum([v for k, v in trans_val.items() if label in str(k).lower()])
                            shifts[label] = abs(orig_conf - trans_conf)

                        for t in CLIP_THRESHOLDS:
                            # Refactored Logic: Accumulate the sum of original continuous confidence deviations
                            total_continuous_shift = sum(s for s in shifts.values() if s >= t)
                            if total_continuous_shift > 0:
                                rec[f'Severity_{t}'] = float(total_continuous_shift)
                                has_any_violation = True
                            else:
                                rec[f'Severity_{t}'] = np.nan

                    elif isinstance(orig_val, (int, float)) and isinstance(trans_val, (int, float)):
                        diff = abs(orig_val - trans_val)
                        for t in CLIP_THRESHOLDS:
                            if diff >= t:
                                rec[f'Severity_{t}'] = float(diff)
                                has_any_violation = True
                            else:
                                rec[f'Severity_{t}'] = np.nan

                    if has_any_violation:
                        records.append(rec)

        except Exception:
            continue

    return records

def get_stat_a12_label(p, a12):
    """Applies conditional filtering: evaluates categorization only when p < 0.05."""
    if p >= 0.05:
        return "No Significant Difference!"
    if a12 >= 0.71: return "Large"
    if a12 >= 0.64: return "Medium"
    if a12 >= 0.56: return "Small"
    return "Negligible"

def run_rq3_analysis(clip_path, blip_path):
    all_records = []
    all_records.extend(extract_vlm_severity_records(clip_path, "CLIP"))
    all_records.extend(extract_vlm_severity_records(blip_path, "BLIP"))

    df_master = pd.DataFrame(all_records)
    if df_master.empty:
        print("Error: No analytical data frames loaded.")
        return

    df_master['Approach'] = df_master['Approach'].str.replace('-', '')

    # =========================================================================
    # PRODUCTION COMPILATION 1: SEPARATE BLIP SUMMARY TABLE
    # =========================================================================
    df_blip_master = df_master[df_master['VLM'] == 'BLIP']
    blip_rows = []

    for order in range(1, 7):
        slice_order = df_blip_master[df_blip_master['Order'] == order]
        nsga_data = slice_order[slice_order['Approach'] == 'NSGAII']['Severity'].values
        rand_data = slice_order[slice_order['Approach'] == 'Random']['Severity'].values

        nsga_mean = np.mean(nsga_data) if len(nsga_data) > 0 else 0.0
        rand_mean = np.mean(rand_data) if len(rand_data) > 0 else 0.0

        if len(nsga_data) > 0 and len(rand_data) > 0:
            try:
                _, p_val = mannwhitneyu(nsga_data, rand_data, alternative='two-sided')
                a12_val = calculate_a12(nsga_data, rand_data)
            except Exception:
                p_val, a12_val = 1.0, 0.5
        else:
            p_val, a12_val = 1.0, 0.5

        blip_rows.append({
            'Order': order,
            'NSGA-II Mean': nsga_mean,
            'Random Mean': rand_mean,
            'A12': get_stat_a12_label(p_val, a12_val)
        })

    df_blip_table = pd.DataFrame(blip_rows)
    print("\n" + "="*95)
    print("      RQ3: BLIP CASCADING METAMORPHIC VIOLATION SEVERITY SUMMARY (OBJECT COUNT DELTAS)      ")
    print("="*95)
    print(df_blip_table.to_string(index=False, formatters={
        'NSGA-II Mean': '{:.1f}'.format, 'Random Mean': '{:.1f}'.format
    }))

    print("\n% --- LaTeX Code for Separate BLIP Summary Table ---")
    latex_blip = df_blip_table.copy()
    for col in ['NSGA-II Mean', 'Random Mean']:
        latex_blip[col] = latex_blip[col].map('{:.1f}'.format)
    print(latex_blip.to_latex(index=False, column_format="crrr", position="th",
                              caption="RQ3: BLIP Cascading Object Detection Mismatch Severity Summary Metrics Across Complexity Orders",
                              label="tab:rq3_blip_severity_summary"))

    # =========================================================================
    # PRODUCTION COMPILATION 2: SEPARATE CLIP MULTI-THRESHOLD SUMMARY TABLE (WIDE MATRIX)
    # =========================================================================
    df_clip_master = df_master[df_master['VLM'] == 'CLIP']
    clip_wide_rows = []

    for order in range(1, 7):
        slice_order = df_clip_master[df_clip_master['Order'] == order]
        row_dict = {'Order': order}

        for t in CLIP_THRESHOLDS:
            nsga_data = slice_order[slice_order['Approach'] == 'NSGAII'][f'Severity_{t}'].dropna().values
            rand_data = slice_order[slice_order['Approach'] == 'Random'][f'Severity_{t}'].dropna().values

            row_dict[f'VLMTest Mean ({t})'] = np.mean(nsga_data) if len(nsga_data) > 0 else 0.0
            row_dict[f'Random Mean ({t})'] = np.mean(rand_data) if len(rand_data) > 0 else 0.0

            if len(nsga_data) > 0 and len(rand_data) > 0:
                try:
                    _, p_val = mannwhitneyu(nsga_data, rand_data, alternative='two-sided')
                    a12_val = calculate_a12(nsga_data, rand_data)
                except Exception:
                    p_val, a12_val = 1.0, 0.5
            else:
                p_val, a12_val = 1.0, 0.5

            row_dict[f'A12 ({t})'] = get_stat_a12_label(p_val, a12_val)

        clip_wide_rows.append(row_dict)

    df_clip_wide = pd.DataFrame(clip_wide_rows)

    print("\n" + "="*195)
    print("                                     RQ3: CLIP MULTI-THRESHOLD CASCADING CONTINUOUS SEVERITY SUMMARY (WIDE HORIZONTAL MATRIX)                                     ")
    print("="*195)

    fmt_clip_dict = {}
    for t in CLIP_THRESHOLDS:
        fmt_clip_dict[f'VLMTest Mean ({t})'] = '{:.3f}'.format
        fmt_clip_dict[f'Random Mean ({t})'] = '{:.3f}'.format

    print(df_clip_wide.to_string(index=False, formatters=fmt_clip_dict))

    print("\n% --- LaTeX Code for Separate CLIP Wide Multi-Threshold Table with Conditional Effect Sizes ---")
    print("\\begin{table}[th]\n\\centering\n\\caption{RQ3: CLIP Multi-Threshold Horizontal Severity Mean Deviation and Categorical Effect Sizes Across Complexity Orders}\n\\label{tab:rq3_clip_wide_continuous_stats}")
    print("\\begin{tabular}{c" + "rrr"*len(CLIP_THRESHOLDS) + "}\n\\toprule")

    top_header = "Order"
    for t in CLIP_THRESHOLDS:
        top_header += f" & \\multicolumn{{3}}{{c}}{{Threshold {t}}}"
    top_header += " \\\\"
    print(top_header)

    sub_lines = ""
    for i in range(len(CLIP_THRESHOLDS)):
        start = 2 + (i*3)
        end = start + 2
        sub_lines += f"\\cmidrule(lr){{{start}-{end}}}"
    print(sub_lines)

    sub_header = " & " + " & ".join(["Mean(V) & Mean(R) & $A_{12}$" for _ in CLIP_THRESHOLDS]) + " \\\\\n\\midrule"
    print(sub_header)

    for _, r in df_clip_wide.iterrows():
        line = f"{int(r['Order'])} & "
        threshold_chunks = []
        for t in CLIP_THRESHOLDS:
            a12_lbl = r[f'A12 ({t})']
            chunk = f"{r[f'VLMTest Mean ({t})']:.3f} & {r[f'Random Mean ({t})']:.3f} & {a12_lbl}"
            threshold_chunks.append(chunk)
        line += " & ".join(threshold_chunks) + " \\\\"
        print(line)

    print("\\bottomrule\n\\end{tabular}\n\\end{table}")

if __name__ == "__main__":
    CLIP_DIR = "/home/user_name/input/"
    BLIP_DIR = "/home/user_name/input/"

    run_rq3_analysis(CLIP_DIR, BLIP_DIR)

In [ ]:
#########################################
# HyperVolume - JMetalpy | Extra Code
########################################

In [ ]:
import os
import re
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np

# Safe import for jMetalPy native quality indicators
try:
    from jmetal.util.quality_indicator import HyperVolume
except ImportError:
    from jmetal.core.quality_indicator import HyperVolume

def extract_non_dominated_front(df, is_nsga=True):
    """
    Extracts relevant solutions and maps them to minimization space.
    - NSGA-II: Isolates the final generation.
    - Random Search: Pools all evaluations to find the best overall front.
    """
    if is_nsga:
        # Isolate the terminal state of the algorithm
        final_gen = df['Generation'].max()
        df_filtered = df[df['Generation'] == final_gen]
    else:
        # Random search keeps all evaluated states to form its best history
        df_filtered = df

    # Map to minimization variables: f1 = -Attack, f2 = Distortion
    raw_pts = df_filtered[['Obj1_Attack_Score', 'Obj2_Distortion_Score']].to_numpy()
    min_space_pts = np.zeros_like(raw_pts)
    min_space_pts[:, 0] = -1.0 * raw_pts[:, 0]
    min_space_pts[:, 1] = raw_pts[:, 1]

    # Simple non-dominated filtering loop
    keep = np.ones(len(min_space_pts), dtype=bool)
    for i in range(len(min_space_pts)):
        for j in range(len(min_space_pts)):
            if i != j and keep[j]:
                if (min_space_pts[j, 0] <= min_space_pts[i, 0] and
                    min_space_pts[j, 1] <= min_space_pts[i, 1] and
                    (min_space_pts[j, 0] < min_space_pts[i, 0] or min_space_pts[j, 1] < min_space_pts[i, 1])):
                    keep[i] = False
                    break

    return min_space_pts[keep]

def compute_framework_hypervolume(root_dir_path, output_csv_name):
    root_path = Path(root_dir_path)
    if not root_path.exists():
        print(f"❌ Error: Root directory path not found: {root_dir_path}")
        return

    csv_files = list(root_path.rglob("*.csv"))
    image_points_pool = defaultdict(list)
    parsed_runs_data = []

    print(f"Parsing optimization logs under {root_path.name}...")
    for file_path in csv_files:
        # Classify approach based on path keywords
        path_str = str(file_path).lower()
        approach = "Random" if "random" in path_str else "NSGA-II"

        # Extract run_id sequence (e.g., run_01)
        match = re.search(r'(run_\d+)', str(file_path))
        run_id = match.group(1) if match else "run_compiled"

        try:
            df = pd.read_csv(file_path)
            required = {'Image_Name', 'Obj1_Attack_Score', 'Obj2_Distortion_Score', 'Generation'}
            if not required.issubset(df.columns):
                continue

            for img_name in df['Image_Name'].unique():
                df_img = df[df['Image_Name'] == img_name]
                front_pts = extract_non_dominated_front(df_img, is_nsga=(approach == "NSGA-II"))

                if len(front_pts) > 0:
                    image_points_pool[img_name].append(front_pts)
                    parsed_runs_data.append({
                        "Run_ID": run_id,
                        "Approach": approach,
                        "Image_Name": img_name,
                        "Front_Points": front_pts
                    })
        except Exception as e:
            print(f"⚠️ Error reading {file_path.name}: {e}")

    # Calculate bounding hypervolume reference points dynamically (Nadir + 5% cushion)
    reference_points_map = {}
    for img_name, list_of_fronts in image_points_pool.items():
        all_pts = np.vstack(list_of_fronts)
        nadir = np.max(all_pts, axis=0)
        ideal = np.min(all_pts, axis=0)
        span = nadir - ideal
        span = np.where(span <= 1e-12, 1.0, span)

        # Add 5% boundary offset to enclose the nadir safely
        ref_point = nadir + 0.05 * span
        reference_points_map[img_name] = [float(ref_point[0]), float(ref_point[1])]

    # Compute Hypervolume using jMetalPy Framework
    hv_records = []
    for record in parsed_runs_data:
        img_name = record["Image_Name"]
        ref_pt = reference_points_map[img_name]

        # Instantiate built-in quality indicator
        hv_indicator = HyperVolume(reference_point=ref_pt)

        try:
            # FIX: Pass the raw 2D NumPy array directly to satisfy moocore's typing requirements
            hv_score = hv_indicator.compute(record["Front_Points"])

            hv_records.append({
                "Run_ID": record["Run_ID"],
                "Approach": record["Approach"],
                "Image_Name": img_name,
                "Hypervolume": float(hv_score),
                "Ref_Point_f1": ref_pt[0],
                "Ref_Point_f2": ref_pt[1]
            })
        except Exception as e:
            print(f"⚠️ Error computing HV for {img_name} in {record['Run_ID']} ({record['Approach']}): {e}")

    # Save to CSV and output stats summary
    if hv_records:
        output_df = pd.DataFrame(hv_records)
        output_df.to_csv(output_csv_name, index=False)
        print(f"✓ Success! File saved as: {output_csv_name}")

        # Print a clear comparison summary breakdown
        summary = output_df.groupby(["Approach"])[["Hypervolume"]].mean()
        print("\n=== Average Hypervolume Summary ===")
        print(summary)
    else:
        print("❌ No matching valid optimization data could be analyzed.")

# ============================================================================
# EXECUTION ENTRY POINT
# ============================================================================
if __name__ == "__main__":
    # Point this to your main working directories containing your run_XX subfolders
    EXPERIMENT_ROOT = "/home/user_name/input/"
    REPORT_OUTPUT = "/home/user_name/input/final_framework_hypervolume_report.csv"

    CLIP_NSGA_ROOT = "/home/user_name/input/"
    CLIP_OUTPUT =  "/home/user_name/input/final_framework_hypervolume_report.csv"


    compute_framework_hypervolume(EXPERIMENT_ROOT, REPORT_OUTPUT)
    compute_framework_hypervolume(CLIP_NSGA_ROOT, CLIP_OUTPUT)